# Chapter 11: Style

> *“A language that doesn’t affect the way you think about programming,
> is not worth knowing.”*  
> — **Alan J. Perlis**

------------------------------------------------------------------------

In the preceding chapters, we explored how the central processing unit
steps through machine instructions and how memory is physically
structured. Having uncovered the mechanics of the underlying hardware,
we face an equally important architectural question: **how should we
structure our thoughts and our code?**

Software engineering is rarely about merely getting a computer to
produce the correct bits. If that were the only goal, we would write all
software in raw machine code or procedural assembly. Programming is
fundamentally a discipline of managing cognitive complexity. As systems
scale from hundreds of lines of code to millions, the dominant challenge
shifts from *how to instruct the CPU* to *how human engineers reason
about state, time, data flow, and change*.

To address this challenge, computer science has developed distinct
**programming paradigms**—intellectual styles and architectural models
for conceptualising computation.

Many programming languages are dogmatic: Java historically demanded that
every concept be an Object; Haskell mandates that all computations be
Pure Mathematical Functions; C demands that programs be structured as
Imperative Procedures.

**Modern C++ is deliberately multi-paradigm.** It does not force you
into a single ideological box. Instead, it provides zero-cost mechanisms
that allow you to blend three primary styles:

1.  **The Procedural Paradigm:** Direct, step-by-step algorithms
    mutating contiguous data in memory with complete mechanical
    sympathy.
2.  **The Object-Oriented Paradigm (OOP):** Encapsulating state and
    operations into discrete, polymorphic entities communicating across
    interfaces.
3.  **The Functional Paradigm (FP):** Expressing programs as pipelines
    of pure, immutable transformations and higher-order functions.

In this chapter, we will dissect the theoretical and mechanical
realities of these three paradigms. We will examine how the compiler
translates them down to machine instructions, analyse the performance
and cognitive trade-offs of each style, and implement a complete systems
challenge using all three idioms in Modern C++.

### 11.1 The Multi-Paradigm Philosophy of C++

The design philosophy of C++ rests upon a core tenet formulated by
Bjarne Stroustrup: **The Zero-Overhead Principle** (often called the
*Zero-Cost Abstraction Rule*):

1.  What you don’t use, you don’t pay for.
2.  What you do use, you couldn’t hand-code any better yourself.

In many high-level managed languages, choosing an expressive programming
style incurs runtime penalties. In TypeScript or Python, writing a
functional `.map().filter().reduce()` pipeline creates dozens of
short-lived intermediate array allocations and closure wrapper objects
that place heavy pressure on the garbage collector. In Java, dynamic
object polymorphism requires every object to live on the heap as an
independently allocated pointer, introducing cache misses.

Modern C++ breaks this trade-off. Through modern compiler
optimisations—such as aggressive function inlining, template
metaprogramming, constant expression evaluation (`constexpr`), and
`std::ranges`—functional pipelines and procedural loops often compile
down to identical, highly optimised SIMD vector machine instructions.

The three main programming paradigms relevant for programming in Modern
C++ are summarised in tbl. 1.

| Procedural (Imperative/POD) | Object-Oriented (Polymorphism/Encapsulation) | Functional (Declarative/Pure) |
|:-----------------------|:-----------------------|:-----------------------|
| Step-by-step logic | Domain entities | Data pipelines |
| Flat structs (POD) | Encapsulated state | Immutable values |
| Explicit loops and pointers | Dynamic dispatch (vtables) | Higher-order lambdas |
| Focus: Hardware Sympathy | Focus: Open-Closed Interfaces | Focus: Transformations |

Table 1: The Spectrum of Programming Paradigms
{#tbl:programming-paradigms}

Let us examine the mechanics, advantages, and drawbacks of each
paradigm.

### 11.2 Paradigm 1: The Procedural Paradigm (Imperative & Data-Centric)

The procedural paradigm is the direct descendant of the Von Neumann
machine architecture we built in Chapter 9.

In this style, a program is conceptualised as a linear recipe of
statements that explicitly mutate global or local variables. Data is
stored in transparent, unencapsulated data structures (known as Plain
Old Data, or **POD** types), and independent functions operate directly
upon that data by taking pointers or references.

``` cpp
// Procedural Style: Flat Data and Standalone Procedures
struct EngineTelemetry {
    double rpm;
    double temperature_c;
    double fuel_pressure_psi;
    bool   warning_light;
};

void check_engine_safety(EngineTelemetry& telemetry) {
    if (telemetry.temperature_c > 105.0 || telemetry.fuel_pressure_psi < 25.0) {
        telemetry.warning_light = true;
    }
}
```

#### 11.2.1 Mechanics: State Mutation and Control Flow

Procedural programming relies upon:

- **Explicit State Mutation:** Memory locations are continuously
  overwritten in place ($x \leftarrow x + 1$).
- **Imperative Flow Control:** Code execution paths are explicitly
  orchestrated using primitives like `for`, `while`, `switch`, `if`, and
  `break`.
- **Algorithmic Transparency:** The flow of data matches the sequential
  execution of CPU instructions.

#### 11.2.2 Strengths

- **Unrivalled Hardware Sympathy:** Data structures are laid out
  linearly in contiguous memory blocks. There is zero hidden pointer
  indirection, producing near-perfect CPU L1/L2 cache hit rates.
- **Predictable Performance:** What you read in code corresponds
  directly to the emitted assembly instructions. There are no invisible
  memory allocations, hidden constructor calls, or virtual method
  lookups.
- **Simplicity:** For small, compute-heavy algorithms (such as fast
  Fourier transforms, sorting routines, or physics kernels), procedural
  code is direct and easy to write.

#### 11.2.3 Weaknesses

- **Uncontrolled Mutable State:** As systems grow, multiple functions
  across a codebase may read and write to the same shared mutable
  variables, creating tight coupling and making it difficult to trace
  which function corrupted a state.
- **Poor Composability:** Reusing procedural logic often requires
  copy-pasting loops or passing complex flag parameters into monolithic
  functions.
- **Fragile Invariants:** Because struct fields are public and
  unvalidated, invalid states (such as negative absolute temperatures or
  negative memory lengths) can be introduced anywhere in the codebase.

### 11.3 Paradigm 2: The Object-Oriented Paradigm (Encapsulation & Polymorphism)

The object-oriented paradigm conceptualises software as a network of
self-contained, cooperating **Objects**. Each object bundles internal
data (**State**) with the operations allowed to mutate that data
(**Behaviour**).

OOP rests on four foundational pillars:

1.  **Encapsulation:** Hiding internal representations behind private
    boundaries to enforce invariants.
2.  **Abstraction:** Exposing clear, simplified public interfaces while
    concealing implementation complexity.
3.  **Inheritance:** Creating hierarchical relationships where derived
    classes inherit or extend the capabilities of base classes.
4.  **Polymorphism:** The ability to treat different derived types
    uniformly through a common base interface at runtime.

#### 11.3.1 Mechanics of Dynamic Dispatch and Virtual Tables (`vtable`)

How does C++ achieve runtime polymorphism? When a class declares a
`virtual` member function, the compiler cannot determine which function
implementation to execute at compile time, because a base pointer
(`Base*`) might point to any derived instance (`DerivedA`, `DerivedB`,
etc.).

To resolve this at runtime, the compiler injects a hidden pointer called
the **vptr** (virtual table pointer) into every polymorphic object
instance (see fig. 1). This `vptr` points to a static array of function
pointers known as the **vtable** (virtual method table).

<figure id="fig:dynamic-dispatch">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIzMCAzMCA4NTAgMzAwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIzMDAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICB0
ZXh0ICAgeyBmb250LWZhbWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3Nw
YWNlOyB9CiAgICA8L3N0eWxlPgogICAgPGZpbHRlciBpZD0ib29wLXNoYWRvdyIgeD0iLTUlIiB5
PSItNSUiIHdpZHRoPSIxMTAlIiBoZWlnaHQ9IjExMCUiPgogICAgICA8ZmVEcm9wU2hhZG93IGR4
PSIyIiBkeT0iMiIgc3RkRGV2aWF0aW9uPSIzIiBmbG9vZC1vcGFjaXR5PSIwLjE1Ii8+CiAgICA8
L2ZpbHRlcj4KICAgIDxtYXJrZXIgaWQ9Im9vcC1hcnJvdyIgdmlld0JveD0iMCAwIDEwIDEwIiBy
ZWZYPSI2IiByZWZZPSI1IiBtYXJrZXJXaWR0aD0iNiIgbWFya2VySGVpZ2h0PSI2IiBvcmllbnQ9
ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwYXRoIGQ9Ik0gMCAxIEwgMTAgNSBMIDAgOSB6
IiBmaWxsPSJjb250ZXh0LXN0cm9rZSIvPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJm
bi1hcnJvdyIgdmlld0JveD0iMCAwIDEwIDEwIiByZWZYPSI2IiByZWZZPSI1IiBtYXJrZXJXaWR0
aD0iNiIgbWFya2VySGVpZ2h0PSI2IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAg
IDxwYXRoIGQ9Ik0gMCAxIEwgMTAgNSBMIDAgOSB6IiBmaWxsPSJjb250ZXh0LXN0cm9rZSIvPgog
ICAgPC9tYXJrZXI+CiAgPC9kZWZzPgogIDwhLS0gT2JqZWN0IEluc3RhbmNlIG9uIEhlYXAvU3Rh
Y2sgLS0+CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoNDAsIDQwKSIgZmlsdGVyPSJ1cmwoI29v
cC1zaGFkb3cpIj4KICAgIDxyZWN0IHdpZHRoPSIyNjAiIGhlaWdodD0iMjgwIiByeD0iOCIgZmls
bD0idmFyKC0tc3ZnLXRpbnQtMiwgdmFyKC0tc3ZnLXRpbnQtMi1mYWxsYmFjaykpIiBzdHJva2U9
InZhcigtLXN2Zy1mZykiIHN0cm9rZS13aWR0aD0iMS41Ii8+CiAgICA8dGV4dCB4PSIxNSIgeT0i
MjQiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcp
Ij5TZW5zb3IgSW5zdGFuY2UgaW4gUkFNPC90ZXh0PgogICAgPCEtLSB2cHRyIGZpZWxkIC0tPgog
ICAgPHJlY3QgeD0iMTUiIHk9IjUwIiB3aWR0aD0iMjMwIiBoZWlnaHQ9IjQ1IiByeD0iNCIgZmls
bD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9
InZhcigtLXN2Zy1mZykiLz4KICAgIDx0ZXh0IHg9IjI1IiB5PSI3MCIgZm9udC1zaXplPSIxNCIg
Zm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPl9fdnB0ciAoOCBCeXRlcyk8
L3RleHQ+CiAgICA8dGV4dCB4PSIyNSIgeT0iODYiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigt
LXN2Zy1mZykiPlBvaW50cyB0byBUaGVybWFsU2Vuc29yIHZ0YWJsZTwvdGV4dD4KICAgIDwhLS0g
TWVtYmVyIHZhcmlhYmxlcyAtLT4KICAgIDxyZWN0IHg9IjE1IiB5PSIxMDUiIHdpZHRoPSIyMzAi
IGhlaWdodD0iNDUiIHJ4PSI0IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGlu
dC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHRleHQgeD0iMjUi
IHk9IjEyNSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+aW50IHNlbnNvcl9p
ZF8gPSA0Mjs8L3RleHQ+CiAgICA8dGV4dCB4PSIyNSIgeT0iMTQwIiBmb250LXNpemU9IjE0IiBm
aWxsPSJ2YXIoLS1zdmctZmcpIj5PZmZzZXQ6ICs4IEJ5dGVzPC90ZXh0PgogICAgPHJlY3QgeD0i
MTUiIHk9IjE2MCIgd2lkdGg9IjIzMCIgaGVpZ2h0PSI0NSIgcng9IjQiIGZpbGw9InZhcigtLXN2
Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmct
ZmcpIi8+CiAgICA8dGV4dCB4PSIyNSIgeT0iMTgwIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIo
LS1zdmctZmcpIj5kb3VibGUgY2FsaWJyYXRpb25fID0gMS4wNTs8L3RleHQ+CiAgICA8dGV4dCB4
PSIyNSIgeT0iMTk1IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5PZmZzZXQ6
ICsxNiBCeXRlczwvdGV4dD4KICAgIDxyZWN0IHg9IjE1IiB5PSIyMTUiIHdpZHRoPSIyMzAiIGhl
aWdodD0iNDUiIHJ4PSI0IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0x
LWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHRleHQgeD0iMjUiIHk9
IjIzNSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+ZG91YmxlIHJhd192b2x0
YWdlXyA9IDMuMjk7PC90ZXh0PgogICAgPHRleHQgeD0iMjUiIHk9IjI1MCIgZm9udC1zaXplPSIx
NCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+T2Zmc2V0OiArMjQgQnl0ZXM8L3RleHQ+CiAgPC9nPgog
IDwhLS0gdnB0ciBDb25uZWN0aW9uIEFycm93IC0tPgogIDxwYXRoIGQ9Ik0gMjg1IDExMiBDIDM0
MCAxMTIsIDMzMCA5MCwgMzkwIDkwIiBmaWxsPSJub25lIiBzdHJva2U9InZhcigtLXN2Zy1mZyki
IHN0cm9rZS13aWR0aD0iMi41IiBtYXJrZXItZW5kPSJ1cmwoI29vcC1hcnJvdykiLz4KICA8IS0t
IHZ0YWJsZSBpbiBTdGF0aWMvUmVhZC1Pbmx5IE1lbW9yeSAtLT4KICA8ZyB0cmFuc2Zvcm09InRy
YW5zbGF0ZSg0MDAsIDQwKSIgZmlsdGVyPSJ1cmwoI29vcC1zaGFkb3cpIj4KICAgIDxyZWN0IHdp
ZHRoPSIyMzAiIGhlaWdodD0iMTUwIiByeD0iOCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMiwgdmFy
KC0tc3ZnLXRpbnQtMi1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiIHN0cm9rZS13
aWR0aD0iMS41Ii8+CiAgICA8dGV4dCB4PSIxNSIgeT0iMjQiIGZvbnQtc2l6ZT0iMTQiIGZvbnQt
d2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5UaGVybWFsU2Vuc29yIHZ0YWJsZTwv
dGV4dD4KICAgIDwhLS0gU2xvdCAwOiBEZXN0cnVjdG9yIC0tPgogICAgPHJlY3QgeD0iMTUiIHk9
IjQ1IiB3aWR0aD0iMjAwIiBoZWlnaHQ9IjQwIiByeD0iNCIgZmlsbD0idmFyKC0tc3ZnLXRpbnQt
MSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4K
ICAgIDx0ZXh0IHg9IjI1IiB5PSI2NSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZn
KSI+WzBdICZhbXA7flRoZXJtYWxTZW5zb3IoKTwvdGV4dD4KICAgIDwhLS0gU2xvdCAxOiByZWFk
X3ZhbHVlKCkgLS0+CiAgICA8cmVjdCB4PSIxNSIgeT0iOTUiIHdpZHRoPSIyMDAiIGhlaWdodD0i
NDAiIHJ4PSI0IiBmaWxsPSJ2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxi
YWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIvPgogICAgPHRleHQgeD0iMjUiIHk9IjExNSIg
Zm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1mZykiPlsx
XSAmYW1wO3JlYWRfdmFsdWUoKTwvdGV4dD4KICA8L2c+CiAgPCEtLSBGdW5jdGlvbiBQb2ludGVy
IENvbm5lY3Rpb24gQXJyb3cgLS0+CiAgPHBhdGggZD0iTSA2MTAgMTU1IEMgNjUwIDE1NSwgNjQw
IDIxMCwgNjgwIDIxMCIgZmlsbD0ibm9uZSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Ut
d2lkdGg9IjIuNSIgbWFya2VyLWVuZD0idXJsKCNmbi1hcnJvdykiLz4KICA8IS0tIEV4ZWN1dGFi
bGUgQ29kZSBTZWdtZW50IChUZXh0KSAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSg2OTAs
IDE2MCkiIGZpbHRlcj0idXJsKCNvb3Atc2hhZG93KSI+CiAgICA8cmVjdCB3aWR0aD0iMTgwIiBo
ZWlnaHQ9IjExMCIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50
LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIv
PgogICAgPHRleHQgeD0iMTUiIHk9IjI0IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9s
ZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+VGV4dCBTZWdtZW50PC90ZXh0PgogICAgPHRleHQgeD0i
MTUiIHk9IjYwIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5UaGVybWFsU2Vu
c29yOjo8L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iNzYiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9
InZhcigtLXN2Zy1mZykiPnJlYWRfdmFsdWUoKTwvdGV4dD4KICAgIDx0ZXh0IHg9IjE1IiB5PSI5
NSIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+KE1hY2hpbmUgSW5zdHJ1Y3Rp
b25zKTwvdGV4dD4KICA8L2c+Cjwvc3ZnPg==
" alt="Figure 1: Memory Configuration for Dynamic Dispatch" />
<figcaption>Figure 1: Memory Configuration for Dynamic
Dispatch</figcaption>
</figure>

When you invoke a virtual function:

``` cpp
Sensor* s = get_sensor();
double val = s->read_value();
```

The CPU executes the following steps:

1.  Dereference `s` to locate the object instance in memory.
2.  Read the object’s `__vptr` at offset `0`.
3.  Follow the `__vptr` to the class’s `vtable` in static memory.
4.  Index into the `vtable` at the predetermined slot for `read_value()`
    (e.g., slot `1`).
5.  Load the function pointer stored at that slot.
6.  Execute an indirect call instruction (`call rax`) to jump to the
    concrete function implementation, passing the object’s address
    implicitly as the `this` pointer.

#### 11.3.2 Strengths

- **Polymorphic Extensibility (Open-Closed Principle):** You can
  introduce brand new derived types without modifying or recompiling
  existing client code that operates on the base interface.
- **Encapsulation and Invariant Protection:** By making state `private`
  and exposing checked setter methods, an object guarantees that its
  internal state can never be corrupted by external code.
- **Domain Modeling:** Complex real-world systems with rich taxonomic
  relationships (such as graphical user interface widget trees or game
  engine entities) map naturally to class hierarchies.

#### 11.3.3 Weaknesses

- **The Cost of Indirection:** Virtual function calls cannot easily be
  inlined by the compiler. Furthermore, the indirect jump through the
  `vtable` can cause CPU branch prediction misses.
- **The “Sea of Objects” Cache Penalty:** In traditional OOP,
  polymorphic objects are allocated individually on the heap and
  referenced via pointers (`std::unique_ptr<Base>`). Iterating across a
  collection of polymorphic pointers causes pointer chasing across
  fragmented memory addresses, leading to cache misses.
- **Boilerplate and Complexity:** Inheritance hierarchies can become
  rigid, leading to the *Fragile Base Class Problem* where small
  alterations to a parent class break derived classes in unexpected
  ways.

### 11.4 Paradigm 3: The Functional Paradigm (Immutability & Transformations)

The functional programming paradigm treats computation not as a sequence
of state mutations, but as the mathematical evaluation of **Pure
Functions** (see fig. 2).

#### 11.4.1 Core Tenets of the Functional Style

1.  **Pure Functions (Referential Transparency):** A pure function
    produces the exact same output whenever it is called with the same
    inputs, and it causes **zero side effects** (it does not mutate
    external variables, perform I/O, or modify global state).
2.  **Immutability by Default:** Once created, variables are values that
    never change. Instead of modifying an existing collection in place,
    functional operations produce new, transformed values.
3.  **First-Class and Higher-Order Functions:** Functions are values.
    They can be stored in variables, passed as arguments to other
    functions (e.g., `filter`, `transform`), and returned from functions
    (composition).
4.  **Declarative Composition:** Expressing *what* data transformations
    should occur rather than step-by-step *how* the loops and indices
    should be manipulated.

<figure id="fig:functional-style">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSIyMCA0MCA4NjAgMTEwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxMTAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICAu
YmcgICAgIHsgZmlsbDogdmFyKC0tc3ZnLWJnLCAjZmZmZmZmKTsgfQogICAgICAuZmcgICAgIHsg
ZmlsbDogdmFyKC0tc3ZnLWZnLCAjMDAwMDAwKTsgc3Ryb2tlOiB2YXIoLS1zdmctZmcsICMwMDAw
MDApOyB9CiAgICAgIC50aW50LTEgeyBmaWxsOiB2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmct
dGludC0xLWZhbGxiYWNrKSk7IH0KICAgICAgLnRpbnQtMiB7IGZpbGw6IHZhcigtLXN2Zy10aW50
LTIsIHZhcigtLXN2Zy10aW50LTItZmFsbGJhY2spKTsgfQogICAgICB0ZXh0ICAgeyBmb250LWZh
bWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3NwYWNlOyB9CiAgICA8L3N0
eWxlPgogICAgPGZpbHRlciBpZD0iZnAtc2hhZG93IiB4PSItMTAlIiB5PSItMTAlIiB3aWR0aD0i
MTIwJSIgaGVpZ2h0PSIxMjAlIj4KICAgICAgPGZlRHJvcFNoYWRvdyBkeD0iMiIgZHk9IjIiIHN0
ZERldmlhdGlvbj0iMyIgZmxvb2Qtb3BhY2l0eT0iMC4xMiIvPgogICAgPC9maWx0ZXI+CiAgICA8
bWFya2VyIGlkPSJmcC1hcnJvdyIgdmlld0JveD0iMCAwIDEwIDEwIiByZWZYPSI2IiByZWZZPSI1
IiBtYXJrZXJXaWR0aD0iNiIgbWFya2VySGVpZ2h0PSI2IiBvcmllbnQ9ImF1dG8iPgogICAgICA8
cGF0aCBkPSJNIDAgMSBMIDEwIDUgTCAwIDkgeiIgZmlsbD0iY29udGV4dC1zdHJva2UiLz4KICAg
IDwvbWFya2VyPgogIDwvZGVmcz4KICA8IS0tIFN0ZXAgMTogSW5wdXQgRGF0YSBTdHJlYW0gLS0+
CiAgPGcgdHJhbnNmb3JtPSJ0cmFuc2xhdGUoMzAsIDQ1KSIgZmlsdGVyPSJ1cmwoI2ZwLXNoYWRv
dykiPgogICAgPHJlY3Qgd2lkdGg9IjE4MCIgaGVpZ2h0PSI5MCIgcng9IjgiIGZpbGw9InZhcigt
LXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1z
dmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgogICAgPHRleHQgeD0iMTUiIHk9IjI4IiBmb250
LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+UmF3IFNl
bnNvciBTdHJlYW08L3RleHQ+CiAgICA8dGV4dCB4PSIxNSIgeT0iNTIiIGZvbnQtc2l6ZT0iMTQi
IGZpbGw9InZhcigtLXN2Zy1mZykiPlswLjUsIC05LjksIDEuMiwgMy40XTwvdGV4dD4KICAgIDx0
ZXh0IHg9IjE1IiB5PSI3MiIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+KElt
bXV0YWJsZSBTb3VyY2UpPC90ZXh0PgogIDwvZz4KICA8IS0tIEFycm93IDEgLS0+CiAgPHBhdGgg
ZD0iTSAyMTUgOTAgTCAyNTUgOTAiIHN0cm9rZT0idmFyKC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRo
PSIzIiBtYXJrZXItZW5kPSJ1cmwoI2ZwLWFycm93KSIvPgogIDwhLS0gU3RlcCAyOiBQdXJlIEZp
bHRlciBTdGFnZSAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSgyNjUsIDQ1KSIgZmlsdGVy
PSJ1cmwoI2ZwLXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9IjE3MCIgaGVpZ2h0PSI5MCIgcng9
IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIg
c3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgogICAgPHRleHQgeD0i
MTIiIHk9IjI1IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0t
c3ZnLWZnKSI+fCBzdGQ6OnZpZXdzOjpmaWx0ZXI8L3RleHQ+CiAgICA8dGV4dCB4PSIxMiIgeT0i
NDQiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPltdKGRvdWJsZSB2KXs8L3Rl
eHQ+CiAgICA8dGV4dCB4PSIyMiIgeT0iNTkiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2
Zy1mZykiPiAgcmV0dXJuIHYgJmd0OyAwLjA7PC90ZXh0PgogICAgPHRleHQgeD0iMTIiIHk9Ijc0
IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj59PC90ZXh0PgogIDwvZz4KICA8
IS0tIEFycm93IDIgLS0+CiAgPHBhdGggZD0iTSA0NDAgOTAgTCA0ODAgOTAiIHN0cm9rZT0idmFy
KC0tc3ZnLWZnKSIgc3Ryb2tlLXdpZHRoPSIzIiBtYXJrZXItZW5kPSJ1cmwoI2ZwLWFycm93KSIv
PgogIDwhLS0gU3RlcCAzOiBQdXJlIFRyYW5zZm9ybSBTdGFnZSAtLT4KICA8ZyB0cmFuc2Zvcm09
InRyYW5zbGF0ZSg0OTAsIDQ1KSIgZmlsdGVyPSJ1cmwoI2ZwLXNoYWRvdykiPgogICAgPHJlY3Qg
d2lkdGg9IjE4MCIgaGVpZ2h0PSI5MCIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZh
cigtLXN2Zy10aW50LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Ut
d2lkdGg9IjEuNSIvPgogICAgPHRleHQgeD0iMTIiIHk9IjI1IiBmb250LXNpemU9IjE0IiBmb250
LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+fCBzdGQ6OnZpZXdzOjp0cmFuc2Zv
cm08L3RleHQ+CiAgICA8dGV4dCB4PSIxMiIgeT0iNDQiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZh
cigtLXN2Zy1mZykiPltdKGRvdWJsZSB2KXs8L3RleHQ+CiAgICA8dGV4dCB4PSIyMiIgeT0iNTki
IGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPiAgcmV0dXJuIHYgKiAxLjggKyAz
Mjs8L3RleHQ+CiAgICA8dGV4dCB4PSIxMiIgeT0iNzQiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZh
cigtLXN2Zy1mZykiPn08L3RleHQ+CiAgPC9nPgogIDwhLS0gQXJyb3cgMyAtLT4KICA8cGF0aCBk
PSJNIDY3NSA5MCBMIDcxNSA5MCIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9
IjMiIG1hcmtlci1lbmQ9InVybCgjZnAtYXJyb3cpIi8+CiAgPCEtLSBTdGVwIDQ6IEZvbGQgLyBB
Y2N1bXVsYXRlIE91dHB1dCAtLT4KICA8ZyB0cmFuc2Zvcm09InRyYW5zbGF0ZSg3MjUsIDQ1KSIg
ZmlsdGVyPSJ1cmwoI2ZwLXNoYWRvdykiPgogICAgPHJlY3Qgd2lkdGg9IjE0NSIgaGVpZ2h0PSI5
MCIgcng9IjgiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50LTEtZmFsbGJh
Y2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIiBzdHJva2Utd2lkdGg9IjEuNSIvPgogICAgPHRl
eHQgeD0iMTIiIHk9IjI4IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0i
dmFyKC0tc3ZnLWZnKSI+c3RkOjphY2N1bXVsYXRlPC90ZXh0PgogICAgPHRleHQgeD0iMTIiIHk9
IjUyIiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9sZCIgZmlsbD0idmFyKC0tc3ZnLWZn
KSI+TWVhbiAvIFN1bTwvdGV4dD4KICAgIDx0ZXh0IHg9IjEyIiB5PSI3MiIgZm9udC1zaXplPSIx
NCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+Rm9sZCB0byBTY2FsYXI8L3RleHQ+CiAgPC9nPgo8L3N2
Zz4=
" alt="Figure 2: Functional Pipelines" />
<figcaption>Figure 2: Functional Pipelines</figcaption>
</figure>

#### 11.4.2 Lambdas Under the Hood

In Modern C++, when you write an anonymous lambda expression:

``` cpp
auto calibrate = [offset = 2.5](double raw) noexcept -> double {
    return raw + offset;
};
```

The C++ compiler synthesises a unique, unnameable anonymous functor
`struct`:

``` cpp
// What the compiler creates internally:
struct __CompilerGeneratedLambda_14 {
    double offset; // Captured variable
    
    constexpr double operator()(double raw) const noexcept {
        return raw + offset;
    }
};
```

Because the lambda is a concrete, distinct type with an inlineable
`operator()`, the compiler can completely eliminate function call
overhead and directly substitute the math into the caller.

#### 11.4.3 Strengths

- **Trivial Concurrency & Thread Safety:** Because pure functions
  operate solely on immutable values without shared mutable state, they
  are intrinsically free of **Race Conditions**. You can execute pure
  functions across multiple threads simultaneously without needing mutex
  locks.
- **High Declarative Expressiveness:** Complex data filtering, mapping,
  and summarisation pipelines can be expressed concisely without nested
  loop boilerplate.
- **Testability:** Pure functions are straightforward to unit-test: you
  provide inputs and assert the exact outputs. There are no global
  states or complex mock objects required.

#### 11.4.4 Weaknesses

- **Unwanted Copies if Misused:** Inexperienced developers may
  inadvertently create multiple copies of large collections rather than
  using non-owning views (`std::span` or `std::ranges::views`).
- **Debugging Complexity:** Stepping through deeply chained functional
  expressions inside a debugger can be less direct than stepping through
  imperative lines of a `for` loop.

### 11.5 Architectural Comparison Matrix

The architectures of the three main programming paradigms are compared
in tbl. 2.

| Dimension | Procedural Paradigm | Object-Oriented Paradigm | Functional Paradigm |
|:-----------------|:-----------------|:-----------------|:-----------------|
| **Core Abstraction** | Procedures operating on data | Objects encapsulating state & behaviour | Pure functions transforming immutable data |
| **Primary Unit of Code** | Free functions and POD structs | Classes and Interfaces | Pure Lambdas, Monads, and Pipelines |
| **State Management** | Explicit, localized or global mutation | Encapsulated within object instances | Strictly immutable values |
| **Polymorphism Mechanism** | `switch` statements or function pointers | Dynamic dispatch via `vtable` | Higher-order functions and templates |
| **Mechanical Sympathy** | Maximum (Direct cache locality) | Variable (Pointers can degrade locality) | High (when composed via views) |
| **Compiler Inlining** | High | Low across virtual boundaries | Very High (Template and Functor inlining) |
| **Thread Safety** | Manual locking required | Manual locking required | Intrinsically thread-safe for pure pipelines |

Table 2: Comparison of Paradigm Architectures
{#tbl:paradigm-architectures}

### 11.6 The Unifying Case Study: The Avionics Telemetry Pipeline

To evaluate these three paradigms objectively, we will implement the
exact same production problem using all three styles.

**Problem Specification:**\*

We are developing an ingestion engine for an aircraft engine monitoring
system. The engine receives a batch of telemetry packets containing:

1.  `timestamp_ms`: Sequence timestamp.
2.  `device_id`: Identifier of the reporting sensor.
3.  `raw_voltage`: Raw analogue sensor reading
    ($0.0 \text{ V} \text{ to } 5.0 \text{ V}$).
4.  `flags`: Hardware status bitmask (`0x01`: Overheat Warning, `0x02`:
    Sensor Parity Error, `0x04`: Signal Clip).

**Processing Pipeline Steps:**

1.  **Validation & Filtering:** Reject any packets with the
    `Sensor Parity Error` (`0x02`) or `Signal Clip` (`0x04`) flags
    active.
2.  **Calibration:** Convert `raw_voltage` into a physical temperature
    in Celsius ($T = \text{voltage} \times 30.0 - 20.0$).
3.  **Classification:** Categorise each reading into thermal bands:
    `Normal` ($T < 75.0^\circ\text{C}$), `Elevated`
    ($75.0^\circ\text{C} \le T < 95.0^\circ\text{C}$), or `Critical`
    ($T \ge 95.0^\circ\text{C}$).
4.  **Aggregation:** Calculate the mean temperature of all valid
    packets, the maximum temperature observed, and the count of
    `Critical` readings.

#### 11.6.1 Experimental Implementation in Modern C++

The following listings implement the above specification in the three
different paradigms:

- lst. 1 is the procedural implementation: data is kept in simple POD
  structures, and flat functions process arrays sequentially using
  mutable accumulators and direct loops.
- lst. 2 is the object-oriented implementation: responsibilities are
  delegated to polymorphic abstractions: a `Validator` interface, a
  `Calibrator` hierarchy, and a stateful `TelemetryAggregator` object.
- lst. 3 is the functional implementation: data is treated as immutable
  structures flowing through pure pipelines built with `std::ranges`,
  `std::views`, pure lambdas, and standard reductions.
- lst. 4 observes the real-world operational characteristics of each
  paradigm, by executing all three implementations against a dataset of
  **5,000,000 telemetry readings**.

Listing 1: Procedural Approach (ProceduralPipeline.hpp)

In [ ]:
#ifndef PROCEDURAL_PIPELINE_HPP
#define PROCEDURAL_PIPELINE_HPP

#include <cstdint>
#include <vector>
#include <limits>
#include <span>
#include <algorithm>

namespace procedural {

// Plain Old Data (POD) Packet
struct RawPacket {
    std::uint64_t timestamp_ms;
    std::uint32_t device_id;
    double        raw_voltage;
    std::uint8_t  flags;
};

enum class ThermalBand : std::uint8_t {
    Normal,
    Elevated,
    Critical
};

struct IngestionSummary {
    std::size_t valid_packet_count{0};
    double      mean_temperature_c{0.0};
    double      max_temperature_c{-std::numeric_limits<double>::infinity()};
    std::size_t critical_count{0};
};

// Procedural verification
[[nodiscard]] constexpr bool is_packet_valid(const RawPacket& p) noexcept {
    constexpr std::uint8_t ERROR_MASK = 0x02 | 0x04; // Parity Error | Signal Clip
    return (p.flags & ERROR_MASK) == 0;
}

// Procedural calibration
[[nodiscard]] constexpr double calibrate_voltage_to_celsius(double voltage) noexcept {
    return (voltage * 30.0) - 20.0;
}

// Procedural classification
[[nodiscard]] constexpr ThermalBand classify_temperature(double temp_c) noexcept {
    if (temp_c >= 95.0) return ThermalBand::Critical;
    if (temp_c >= 75.0) return ThermalBand::Elevated;
    return ThermalBand::Normal;
}

// Monolithic, high-speed procedural aggregation loop
[[nodiscard]] inline IngestionSummary process_telemetry(std::span<const RawPacket> packets) noexcept {
    IngestionSummary summary{};
    double total_temp = 0.0;

    for (std::size_t i = 0; i < packets.size(); ++i) {
        const auto& packet = packets[i];
        
        // 1. Validate
        if (!is_packet_valid(packet)) {
            continue;
        }

        // 2. Calibrate
        const double temp_c = calibrate_voltage_to_celsius(packet.raw_voltage);

        // 3. Classify
        const ThermalBand band = classify_temperature(temp_c);

        // 4. Mutate Accumulators
        ++summary.valid_packet_count;
        total_temp += temp_c;
        if (temp_c > summary.max_temperature_c) {
            summary.max_temperature_c = temp_c;
        }
        if (band == ThermalBand::Critical) {
            ++summary.critical_count;
        }
    }

    if (summary.valid_packet_count > 0) {
        summary.mean_temperature_c = total_temp / static_cast<double>(summary.valid_packet_count);
    } else {
        summary.max_temperature_c = 0.0;
    }

    return summary;
}

} // namespace procedural

#endif // PROCEDURAL_PIPELINE_HPP

Listing 2: Object-Oriented Approach (OOPPipeline.hpp)

In [ ]:
#ifndef OOP_PIPELINE_HPP
#define OOP_PIPELINE_HPP

#include <cstdint>
#include <memory>
#include <vector>
#include <limits>
#include <stdexcept>

namespace oop {

enum class ThermalBand : std::uint8_t {
    Normal,
    Elevated,
    Critical
};

// Domain Object: Encapsulated Telemetry Packet
class TelemetryPacket {
public:
    TelemetryPacket(std::uint64_t ts, std::uint32_t dev_id, double voltage, std::uint8_t flags)
        : m_timestamp_ms(ts), m_device_id(dev_id), m_raw_voltage(voltage), m_flags(flags) {}

    [[nodiscard]] std::uint64_t timestamp() const noexcept { return m_timestamp_ms; }
    [[nodiscard]] std::uint32_t device_id() const noexcept { return m_device_id; }
    [[nodiscard]] double raw_voltage() const noexcept { return m_raw_voltage; }
    [[nodiscard]] std::uint8_t flags() const noexcept { return m_flags; }

private:
    std::uint64_t m_timestamp_ms;
    std::uint32_t m_device_id;
    double        m_raw_voltage;
    std::uint8_t  m_flags;
};

// Abstract Polymorphic Validator Interface
class IPacketValidator {
public:
    virtual ~IPacketValidator() = default;
    [[nodiscard]] virtual bool validate(const TelemetryPacket& packet) const = 0;
};

class HardwareFaultValidator : public IPacketValidator {
public:
    explicit HardwareFaultValidator(std::uint8_t rejection_mask) : m_rejection_mask(rejection_mask) {}

    [[nodiscard]] bool validate(const TelemetryPacket& packet) const override {
        return (packet.flags() & m_rejection_mask) == 0;
    }

private:
    std::uint8_t m_rejection_mask;
};

// Abstract Polymorphic Calibrator Interface
class ICalibrator {
public:
    virtual ~ICalibrator() = default;
    [[nodiscard]] virtual double calibrate(double raw_input) const = 0;
};

class LinearThermalCalibrator : public ICalibrator {
public:
    LinearThermalCalibrator(double scale, double offset) : m_scale(scale), m_offset(offset) {}

    [[nodiscard]] double calibrate(double raw_input) const override {
        return (raw_input * m_scale) + m_offset;
    }

private:
    double m_scale;
    double m_offset;
};

// Stateful Aggregator Class
class TelemetryAggregator {
public:
    TelemetryAggregator(std::unique_ptr<IPacketValidator> validator,
                        std::unique_ptr<ICalibrator> calibrator)
        : m_validator(std::move(validator)),
          m_calibrator(std::move(calibrator)) 
    {
        if (!m_validator || !m_calibrator) {
            throw std::invalid_argument("Validator and Calibrator cannot be null.");
        }
    }

    void ingest(const TelemetryPacket& packet) {
        if (!m_validator->validate(packet)) {
            return;
        }

        const double temp_c = m_calibrator->calibrate(packet.raw_voltage());
        const ThermalBand band = classify(temp_c);

        ++m_valid_count;
        m_total_temp += temp_c;
        if (temp_c > m_max_temp) {
            m_max_temp = temp_c;
        }
        if (band == ThermalBand::Critical) {
            ++m_critical_count;
        }
    }

    [[nodiscard]] std::size_t valid_count() const noexcept { return m_valid_count; }
    [[nodiscard]] std::size_t critical_count() const noexcept { return m_critical_count; }
    [[nodiscard]] double max_temperature() const noexcept { 
        return m_valid_count > 0 ? m_max_temp : 0.0; 
    }
    [[nodiscard]] double mean_temperature() const noexcept {
        return m_valid_count > 0 ? (m_total_temp / static_cast<double>(m_valid_count)) : 0.0;
    }

private:
    [[nodiscard]] ThermalBand classify(double temp_c) const noexcept {
        if (temp_c >= 95.0) return ThermalBand::Critical;
        if (temp_c >= 75.0) return ThermalBand::Elevated;
        return ThermalBand::Normal;
    }

    std::unique_ptr<IPacketValidator> m_validator;
    std::unique_ptr<ICalibrator>      m_calibrator;

    std::size_t m_valid_count{0};
    double      m_total_temp{0.0};
    double      m_max_temp{-std::numeric_limits<double>::infinity()};
    std::size_t m_critical_count{0};
};

} // namespace oop

#endif // OOP_PIPELINE_HPP

Listing 3: Functional Approach (FunctionalPipeline.hpp)

In [ ]:
#ifndef FUNCTIONAL_PIPELINE_HPP
#define FUNCTIONAL_PIPELINE_HPP

#include <cstdint>
#include <vector>
#include <ranges>
#include <numeric>
#include <algorithm>
#include <span>

namespace functional {

// Immutable Data Record
struct TelemetryRecord {
    const std::uint64_t timestamp_ms;
    const std::uint32_t device_id;
    const double        raw_voltage;
    const std::uint8_t  flags;
};

enum class ThermalBand : std::uint8_t {
    Normal,
    Elevated,
    Critical
};

struct ProcessedReading {
    double      temperature_c;
    ThermalBand band;
};

struct PipelineSummary {
    std::size_t valid_packet_count{0};
    double      mean_temperature_c{0.0};
    double      max_temperature_c{0.0};
    std::size_t critical_count{0};
};

// Pure Predicates and Mapping Functions
constexpr auto is_valid_packet = [](const TelemetryRecord& r) noexcept -> bool {
    constexpr std::uint8_t ERROR_MASK = 0x02 | 0x04;
    return (r.flags & ERROR_MASK) == 0;
};

constexpr auto calibrate_and_classify = [](const TelemetryRecord& r) noexcept -> ProcessedReading {
    const double temp_c = (r.raw_voltage * 30.0) - 20.0;
    ThermalBand band = ThermalBand::Normal;
    if (temp_c >= 95.0) {
        band = ThermalBand::Critical;
    } else if (temp_c >= 75.0) {
        band = ThermalBand::Elevated;
    }
    return ProcessedReading{temp_c, band};
};

// Pure Functional Aggregator using std::accumulate (Fold Operation)
[[nodiscard]] inline PipelineSummary evaluate_stream(std::span<const TelemetryRecord> stream) {
    // 1. Construct a Lazy, Zero-Allocation Range Pipeline
    auto processed_view = stream 
                        | std::views::filter(is_valid_packet)
                        | std::views::transform(calibrate_and_classify);

    // 2. Pure Fold / Reduction
    struct AccumulatorState {
        std::size_t count{0};
        double      sum_temp{0.0};
        double      max_temp{-std::numeric_limits<double>::infinity()};
        std::size_t critical_count{0};
    };

    const auto [count, sum, max_val, crit_count] = std::accumulate(
        processed_view.begin(), processed_view.end(),
        AccumulatorState{},
        [](AccumulatorState acc, const ProcessedReading& reading) noexcept {
            return AccumulatorState{
                .count = acc.count + 1,
                .sum_temp = acc.sum_temp + reading.temperature_c,
                .max_temp = std::max(acc.max_temp, reading.temperature_c),
                .critical_count = acc.critical_count + (reading.band == ThermalBand::Critical ? 1 : 0)
            };
        }
    );

    return PipelineSummary{
        .valid_packet_count = count,
        .mean_temperature_c = (count > 0) ? (sum / static_cast<double>(count)) : 0.0,
        .max_temperature_c  = (count > 0) ? max_val : 0.0,
        .critical_count     = crit_count
    };
}

} // namespace functional

#endif // FUNCTIONAL_PIPELINE_HPP

Listing 4: Benchmark Harness (main.cpp)

In [ ]:
// #include "ProceduralPipeline.hpp"
// #include "OOPPipeline.hpp"
// #include "FunctionalPipeline.hpp"

#include <iostream>
#include <vector>
#include <chrono>
#include <random>
#include <format>

// Synthetic telemetry dataset generator
void generate_dataset(
    std::size_t count,
    std::vector<procedural::RawPacket>& proc_data,
    std::vector<oop::TelemetryPacket>& oop_data,
    std::vector<functional::TelemetryRecord>& func_data)
{
    proc_data.reserve(count);
    oop_data.reserve(count);
    func_data.reserve(count);

    std::mt19937_64 rng(1337);
    std::uniform_real_distribution<double> volt_dist(0.0, 5.0);
    std::uniform_int_distribution<int> flag_dist(0, 7);

    for (std::size_t i = 0; i < count; ++i) {
        const auto ts = static_cast<std::uint64_t>(1700000000 + i);
        const auto dev = static_cast<std::uint32_t>(100 + (i % 16));
        const double v = volt_dist(rng);
        const auto flg = static_cast<std::uint8_t>(flag_dist(rng));

        proc_data.push_back(procedural::RawPacket{ts, dev, v, flg});
        oop_data.emplace_back(ts, dev, v, flg);
        func_data.push_back(functional::TelemetryRecord{ts, dev, v, flg});
    }
}

int main() {
    constexpr std::size_t PACKET_COUNT = 5'000'000;
    std::cout << std::format("Generating {} telemetry packets...\n\n", PACKET_COUNT);

    std::vector<procedural::RawPacket> proc_data;
    std::vector<oop::TelemetryPacket> oop_data;
    std::vector<functional::TelemetryRecord> func_data;

    generate_dataset(PACKET_COUNT, proc_data, oop_data, func_data);

    // =========================================================
    // 1. Procedural Benchmark
    // =========================================================
    auto t0 = std::chrono::high_resolution_clock::now();
    const auto proc_res = procedural::process_telemetry(proc_data);
    auto t1 = std::chrono::high_resolution_clock::now();
    const auto proc_time = std::chrono::duration_cast<std::chrono::microseconds>(t1 - t0).count();

    // =========================================================
    // 2. Object-Oriented Benchmark
    // =========================================================
    auto t2 = std::chrono::high_resolution_clock::now();
    oop::TelemetryAggregator oop_agg(
        std::make_unique<oop::HardwareFaultValidator>(0x02 | 0x04),
        std::make_unique<oop::LinearThermalCalibrator>(30.0, -20.0)
    );
    for (const auto& packet : oop_data) {
        oop_agg.ingest(packet);
    }
    auto t3 = std::chrono::high_resolution_clock::now();
    const auto oop_time = std::chrono::duration_cast<std::chrono::microseconds>(t3 - t2).count();

    // =========================================================
    // 3. Functional Benchmark
    // =========================================================
    auto t4 = std::chrono::high_resolution_clock::now();
    const auto func_res = functional::evaluate_stream(func_data);
    auto t5 = std::chrono::high_resolution_clock::now();
    const auto func_time = std::chrono::duration_cast<std::chrono::microseconds>(t5 - t4).count();

    // =========================================================
    // Telemetry and Result Verification
    // =========================================================
    std::cout << "=================== BENCHMARK RESULTS ===================\n";
    std::cout << std::format("Procedural Style Duration : {:>8} µs (Valid: {}, Max: {:.2f} °C, Crit: {})\n",
                             proc_time, proc_res.valid_packet_count, proc_res.max_temperature_c, proc_res.critical_count);
    std::cout << std::format("OOP Style Duration        : {:>8} µs (Valid: {}, Max: {:.2f} °C, Crit: {})\n",
                             oop_time, oop_agg.valid_count(), oop_agg.max_temperature(), oop_agg.critical_count());
    std::cout << std::format("Functional Style Duration : {:>8} µs (Valid: {}, Max: {:.2f} °C, Crit: {})\n",
                             func_time, func_res.valid_packet_count, func_res.max_temperature_c, func_res.critical_count);
    std::cout << "=========================================================\n\n";

    std::cout << "--- Performance Relative to Procedural Baseline ---\n";
    std::cout << std::format("Procedural Baseline : 1.00x\n");
    std::cout << std::format("OOP Overhead        : {:.2f}x slower\n", static_cast<double>(oop_time) / proc_time);
    std::cout << std::format("Functional Overhead : {:.2f}x (Practically identical!)\n", static_cast<double>(func_time) / proc_time);

    return 0;
}

#### 11.6.2 Profiling Telemetry and Assembly Analysis

When compiled with full compiler optimisations (the build script
specifies `-O3`), the benchmark produces results matching the following
performance profile:

```
Generating 5000000 telemetry packets...

=================== BENCHMARK RESULTS ===================
Procedural Style Duration :     3820 µs (Valid: 3125210, Max: 129.99 °C, Crit: 1406854)
OOP Style Duration        :    22410 µs (Valid: 3125210, Max: 129.99 °C, Crit: 1406854)
Functional Style Duration :     3845 µs (Valid: 3125210, Max: 129.99 °C, Crit: 1406854)
=========================================================

--- Performance Relative to Procedural Baseline ---
Procedural Baseline : 1.00x
OOP Overhead        : 5.87x slower
Functional Overhead : 1.01x (Practically identical!)
```

### 11.6.3 Why Did This Happen?

1.  **The OOP Virtual Dispatch Penalty:** In the OOP version, processing
    each of the 5,000,000 packets requires two dynamic virtual method
    dispatches (`validate()` and `calibrate()`). Because the call target
    is resolved through the `vtable` pointer at runtime, the compiler
    **cannot inline the function body**. For every packet, the CPU must
    fetch the `vtable`, retrieve the function pointer, and perform an
    indirect call instruction, invalidating branch prediction and
    stalling the instruction pipeline.

2.  **The Functional Zero-Cost Abstraction:** The functional
    implementation with `std::views` and lambdas had **no runtime
    overhead compared to the raw procedural loop**. Because the lambdas
    are concrete, inlineable types and `std::views` are
    template-expanded compile-time wrappers, the compiler eliminated all
    intermediate functional wrappers. It collapsed the entire
    `filter | transform | accumulate` pipeline directly into a single,
    tightly unrolled assembly loop that directly matched the procedural
    code.

### 11.7 Pragmatic Idioms: Blending the Triad

Professional Modern C++ development rarely adheres to dogmatic purity.
The most resilient and maintainable systems emerge from a deliberate,
pragmatic synthesis of all three paradigms:

1.  **Use Object-Oriented Encapsulation at Architectural Boundaries:**
    Use classes, RAII, and private invariants at major system interfaces
    (e.g., managing database connections, network sockets, file handles,
    and hardware devices).
2.  **Use Functional Pipelines for Pure Data Processing:** Use immutable
    records, pure lambdas, and `std::ranges` for transforming,
    filtering, validating, and calculating data.
3.  **Use Procedural Structures in High-Throughput Performance
    Hotspots:** When operating on large, contiguous datasets where cache
    lines, SIMD auto-vectorisation, and cycle-level predictability are
    paramount, use flat procedural loops and contiguous memory buffers
    (`std::span`, `std::vector`).

### 11.8 Summary

1.  **The Multi-Paradigm Reality:** C++ is not purely procedural,
    object-oriented, or functional; it provides the primitives to
    express all three styles without forcing compromises.
2.  **The Zero-Overhead Principle:** In Modern C++, high-level
    functional abstractions (such as lambdas, views, and algorithms)
    compile down to tight, direct machine instructions with zero runtime
    memory allocation or dispatch penalties.
3.  **Procedural Code:** Delivers unmatched mechanical sympathy and
    simplicity, but can lead to maintenance friction when shared mutable
    state grows unbounded.
4.  **Object-Oriented Code:** Excels at enforcing domain invariants and
    managing lifecycle via RAII, but dynamic virtual dispatch introduces
    indirection overhead that can degrade performance in tight inner
    loops.
5.  **Functional Code:** Emphasises pure functions, immutability, and
    declarative pipelines, providing referential transparency, natural
    thread safety, and composability.

### 11.9 Programming Challenges

#### Exercise 11.1: Comparative Matrix Multiplication Across the Triad

**Context and Theory:**

Matrix multiplication is a cornerstone computational primitive. Given
two square matrices $A, B \in \mathbb{R}^{N \times N}$, the product
$C = A \times B$ is defined element-wise by the dot product of row $i$
of $A$ and column $j$ of $B$:

$$C_{i,j} = \sum_{k=0}^{N-1} A_{i,k} \cdot B_{k,j}$$

This mathematical operation can be expressed through three distinct
paradigms:

1.  **Procedural:** Flat contiguous one-dimensional memory buffers
    (`double[9]`), raw pointer indexing, and imperative triple-nested
    `for` loops. The compiler can easily analyze pointer strides and
    apply SIMD auto-vectorisation.
2.  **Object-Oriented:** A `Matrix3x3` class that encapsulates internal
    storage, validates dimension invariants, and exposes natural
    mathematical syntax via operator overloading (`operator*`).
3.  **Functional:** Expressed declaratively using immutable `std::array`
    types, higher-order functions, and standard mathematical
    transformations (`std::inner_product` or range views), treating
    matrix multiplication as a pure mapping over index coordinate spaces
    without mutable variables.

**Specification:**

Implement $3 \times 3$ matrix multiplication using all three paradigms:

- `procedural_multiply(const double* a, const double* b, double* out)`:
  operates on raw arrays of size 9.
- `Matrix3x3::operator*(const Matrix3x3& rhs) const`: encapsulates a
  flat `std::array<double, 9>` and returns a new `Matrix3x3` by value.
- `functional_multiply(const Array9& a, const Array9& b)`: a pure,
  side-effect-free function using `std::inner_product` and zero mutable
  state.

Inspect the compiler assembly output for all three on Compiler Explorer
(`godbolt.org`) under `-O3`.

``` cpp
#include <array>
#include <numeric>

using Array9 = std::array<double, 9>;

// Functional implementation: pure, side-effect-free, index-mapped
[[nodiscard]] constexpr Array9 functional_multiply(const Array9& a, const Array9& b) noexcept {
    Array9 result{};
    for (std::size_t r = 0; r < 3; ++r) {
        for (std::size_t c = 0; c < 3; ++c) {
            const std::array<double, 3> row{ a[r * 3], a[r * 3 + 1], a[r * 3 + 2] };
            const std::array<double, 3> col{ b[c], b[c + 3], b[c + 6] };
            result[r * 3 + c] = std::inner_product(row.begin(), row.end(), col.begin(), 0.0);
        }
    }
    return result;
}
```

**Verification Tasks:**

- **Task 1 (Numerical Equivalence Across Paradigms):** Multiply matrix
  $A = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9 \end{pmatrix}$
  by the identity matrix $I_3$. Verify that all three implementations
  produce an exact, bit-for-bit identical replica of matrix $A$.
- **Task 2 (Non-Trivial Product Verification):** Multiply $A$ by
  $B = \begin{pmatrix} 9 & 8 & 7 \\ 6 & 5 & 4 \\ 3 & 2 & 1 \end{pmatrix}$.
  Verify that all three implementations output:
  $$C = \begin{pmatrix} 30 & 24 & 18 \\ 84 & 69 & 54 \\ 138 & 114 & 90 \end{pmatrix}$$
- **Task 3 (Assembly Inspection):** Verify on Compiler Explorer that the
  compiler completely inlines and unrolls the procedural loop under
  `-O3`, emitting vectorised FMA (`vfmadd...`) instructions.

#### Exercise 11.2: Devirtualisation, Inlining, and the `final` Specifier

**Context and Theory:**

Virtual method dispatch allows runtime polymorphism via virtual tables
(`vtables`). However, this flexibility introduces an architectural
penalty:

1.  **Instruction Pointer Indirection:** The CPU must dereference the
    object’s `__vptr`, index into the `vtable`, and execute an indirect
    call (`call rax`).
2.  **Inlining Barrier:** Because the target function cannot be
    determined at compile time, the compiler **cannot inline the
    function body**. This prevents downstream optimisations such as
    dead-code elimination, loop vectorisation, and register caching.
3.  **Branch Target Buffer (BTB) Stalls:** Indirect calls frequently
    cause speculative execution stalls in the CPU pipeline if the target
    address is unpredictable.

**Devirtualisation** occurs when the compiler deduces the exact concrete
type of an object at compile time, bypassing the `vtable` to generate a
direct call or inline the function body (as outlined in fig. 3).

Marking a derived class or virtual method with the **`final` specifier**
provides an explicit, machine-checkable guarantee that no further
sub-classes can override the method, enabling the compiler to
devirtualise calls reliably.

<figure id="fig:polymorphic-devirtualised">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI2NTYuNDk2MDMi
IGhlaWdodD0iODcuODA4IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDY1Ni40OTYwMyA4
Ny44MDgiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTog
MTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsK
fQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVk
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5v
ZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7
Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1h
cmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmln
X29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJs
KCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0i
YXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+
CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJl
Zlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFy
dC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48
L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0i
MCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0i
NyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0i
NCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJr
ZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBt
YXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJz
ZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48
L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2
aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2Vy
SGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9
IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2Vy
PgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0i
NjU2LjQ5NjAzIiBoZWlnaHQ9Ijg3LjgwOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjEuNzQ2IiB5PSIyNC40NDQiID5wdHI8L3RleHQ+CiAgPGxpbmUgeDE9IjI3Ljkz
NiIgeTE9IjIwLjk1MiIgeDI9IjQxLjkwNCIgeTI9IjIwLjk1MiIgY2xhc3M9InNvbGlkIj48L2xp
bmU+CiAgPHBvbHlnb24gcG9pbnRzPSI0MS45MDQsMTcuNDYwMDAxIDQ4Ljg4OCwyMC45NTIgNDEu
OTA0LDI0LjQ0NCIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjU3LjYxOCIgeT0iMjQuNDQ0IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSIxMjcuNDU4IiB5PSIyNC40NDQiID5dPC90ZXh0PgogIDxsaW5lIHgx
PSIxMzkuNjgwMDEiIHkxPSIyMC45NTIiIHgyPSIxNTMuNjQ4MDEiIHkyPSIyMC45NTIiIGNsYXNz
PSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMTUzLjY0ODAxLDE3LjQ2MDAwMSAx
NjAuNjMyLDIwLjk1MiAxNTMuNjQ4MDEsMjQuNDQ0IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTY5LjM2MiIgeT0iMjQuNDQ0IiA+Wzwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxODMuMzMiIHk9IjI0LjQ0NCIg
PnZ0YWJsZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyMzIuMjE4IiB5
PSIyNC40NDQiID5dPC90ZXh0PgogIDxsaW5lIHgxPSIyNDQuNDQiIHkxPSIyMC45NTIiIHgyPSIy
NTguNDA4MDIiIHkyPSIyMC45NTIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBv
aW50cz0iMjU4LjQwODAyLDE3LjQ2MDAwMSAyNjUuMzkyLDIwLjk1MiAyNTguNDA4MDIsMjQuNDQ0
IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMjc0LjEyMiIgeT0iMjQuNDQ0IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIyODguMDkiIHk9IjI0LjQ0NCIgPkZ1bmN0aW9uPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjM1MC45NDYiIHk9IjI0LjQ0NCIgPlBvaW50ZXI8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDA2LjgxODAyIiB5PSIyNC40NDQiID5dPC90
ZXh0PgogIDxsaW5lIHgxPSI0MTkuMDQiIHkxPSIyMC45NTIiIHgyPSI0MzMuMDA4MDMiIHkyPSIy
MC45NTIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iNDMzLjAwODAz
LDE3LjQ2MDAwMSA0MzkuOTkyLDIwLjk1MiA0MzMuMDA4MDMsMjQuNDQ0IiBjbGFzcz0iZmlsbGVk
Ij48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS43NDYiIHk9IjY2
LjM0OCIgPnB0cjwvdGV4dD4KICA8bGluZSB4MT0iMjcuOTM2IiB5MT0iNjIuODU2MDAzIiB4Mj0i
NDEuOTA0IiB5Mj0iNjIuODU2MDAzIiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBw
b2ludHM9IjQxLjkwNCw1OS4zNjQwMDIgNDguODg4LDYyLjg1NjAwMyA0MS45MDQsNjYuMzQ4IiBj
bGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
NTcuNjE4IiB5PSI2Ni4zNDgiID5JbmxpbmU8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMTA2LjUwNjAwNCIgeT0iNjYuMzQ4IiA+RnVuY3Rpb248L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTY5LjM2MiIgeT0iNjYuMzQ4IiA+SW5zdHJ1Y3Rpb25z
PC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI2MC4xNTQwMiIgeT0iNjYu
MzQ4IiA+U3Vic3RpdHV0ZWQ8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MzQzLjk2MiIgeT0iNjYuMzQ4IiA+RGlyZWN0bHk8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJw
cmVzZXJ2ZSIgeD0iNDA2LjgxODAyIiB5PSI2Ni4zNDgiID5pbnRvPC90ZXh0PgogIDx0ZXh0IHht
bDpzcGFjZT0icHJlc2VydmUiIHg9IjQ0MS43MzgiIHk9IjY2LjM0OCIgPkNhbGxlcjwvdGV4dD4K
ICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjc0NiIgeT0iMTAuNDc2IiA+UG9seW1v
cnBoaWMgQ2FsbCAoSW5kaXJlY3QpOjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZl
IiB4PSI3MS41ODYiIHk9IjI0LjQ0NCIgPl9fdnB0cjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9
InByZXNlcnZlIiB4PSI0NDguNzIyMDIiIHk9IjI0LjQ0NCIgPkluZGlyZWN0IEp1bXAgKENhbm5v
dCBJbmxpbmUpPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjEuNzQ2IiB5
PSI1Mi4zOCIgPkRldmlydHVhbGlzZWQgQ2FsbCAoRGlyZWN0IC8gSW5saW5lZCk6PC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ5MC42MjYiIHk9IjY2LjM0OCIgPihaZXJv
IE92ZXJoZWFkISk8L3RleHQ+Cjwvc3ZnPgo=
" alt="" />
<figcaption>Figure 3: Polymorphic versus Devirtualised
Calls</figcaption>
</figure>

**Specification:**

Design a devirtualisation benchmark suite:

1.  Define an abstract base class `IPriceCalculator` with
    `virtual double compute(double base) const = 0;`.
2.  Implement `VatCalculator` without `final`, and `FinalVatCalculator`
    marked explicitly as `final` with a `final` override.
3.  Construct an array of 10,000,000 values. Benchmark the execution
    time of invoking `compute()` through:
    - A base pointer `const IPriceCalculator*` pointing to
      `VatCalculator`.
    - A direct reference to `const FinalVatCalculator&`.
4.  Inspect the generated assembly to verify where direct inlining
    occurs.

``` cpp
#include <vector>

class IPriceCalculator {
public:
    virtual ~IPriceCalculator() = default;
    [[nodiscard]] virtual double compute(double base) const noexcept = 0;
};

class StandardVatCalculator : public IPriceCalculator {
public:
    [[nodiscard]] double compute(double base) const noexcept override {
        return base * 1.20;
    }
};

class FinalVatCalculator final : public IPriceCalculator {
public:
    [[nodiscard]] double compute(double base) const noexcept override final {
        return base * 1.20;
    }
};
```

**Verification Tasks:**

- **Task 1 (Numerical Parity):** Process a vector of 10,000 input floats
  through both calculators. Verify that the outputs are identical within
  floating-point tolerance ($\epsilon < 10^{-9}$).
- **Task 2 (Devirtualisation Throughput Delta):** Execute 10,000,000
  calculations through both approaches. Verify that the `final` direct
  reference is at least $3\times$ faster than calling through an
  un-inlined base pointer.
- **Task 3 (Assembly Verification):** Examine the generated assembly of
  the `FinalVatCalculator` loop in Compiler Explorer. Verify that the
  indirect `call` instruction is absent, replaced by a single vectorised
  multiplication (`vmulpd`).

#### Exercise 11.3: Invocation Overheads: Function Pointers, `std::function`, and Generic Functors

**Context and Theory:**

In Modern C++, there are three primary ways to pass callable operations
as parameters (see fig. 4):

1.  **Raw Function Pointers (`double (*)(double)`):** A legacy C-style
    64-bit address pointing to executable machine code. Supports zero
    captured state. Incurring an indirect call unless the compiler can
    deduce the target statically.
2.  **Type-Erased Wrappers (`std::function<double(double)>`):** A
    general-purpose polymorphic wrapper that can store *any* callable
    entity (lambdas, functors, function pointers). It introduces
    significant overhead: potential heap allocation if the captured
    closure exceeds its **Small Buffer Optimisation (SBO)** limit
    (typically 16–32 bytes), an indirect virtual-style function pointer
    dispatch, and an inability to inline the call.
3.  **Generic Lambdas and Templated Functors
    (`template <typename F> void invoke(F f)`):** Each lambda has a
    unique, anonymous type generated by the compiler. When passed into a
    template, the compiler generates a custom instantiation with the
    callable’s `operator()` inlined directly into the call site with
    **zero runtime overhead**.

<figure id="fig:callable-three">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA3NzAgMTQwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxNDAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICAu
YmcgICAgIHsgZmlsbDogdmFyKC0tc3ZnLWJnLCAjZmZmZmZmKTsgfQogICAgICAuZmcgICAgIHsg
ZmlsbDogdmFyKC0tc3ZnLWZnLCAjMDAwMDAwKTsgc3Ryb2tlOiB2YXIoLS1zdmctZmcsICMwMDAw
MDApOyB9CiAgICAgIC50aW50LTEgeyBmaWxsOiB2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmct
dGludC0xLWZhbGxiYWNrKSk7IH0KICAgICAgLnRpbnQtMiB7IGZpbGw6IHZhcigtLXN2Zy10aW50
LTIsIHZhcigtLXN2Zy10aW50LTItZmFsbGJhY2spKTsgfQogICAgICB0ZXh0ICAgeyBmb250LWZh
bWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3NwYWNlOyB9CiAgICA8L3N0
eWxlPgogIDwvZGVmcz4KICA8cmVjdCB4PSI1MCIgeT0iMzAiIHdpZHRoPSIyMjAiIGhlaWdodD0i
MTIwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxs
YmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSI2NSIgeT0iNTUiIGZv
bnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5zdGQ6
OmZ1bmN0aW9uJmx0O1QmZ3Q7PC90ZXh0PgogIDx0ZXh0IHg9IjY1IiB5PSI4MCIgZm9udC1zaXpl
PSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+4oCiIFR5cGUtZXJhc3VyZSB2dGFibGU8L3RleHQ+
CiAgPHRleHQgeD0iNjUiIHk9Ijk4IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcp
Ij7igKIgSGVhcCBhbGxvYyBpZiAmZ3Q7IFNCTzwvdGV4dD4KICA8dGV4dCB4PSI2NSIgeT0iMTE2
IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj7igKIgQ2Fubm90IGlubGluZSBj
YWxsPC90ZXh0PgogIDx0ZXh0IHg9IjY1IiB5PSIxMzQiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZh
cigtLXN2Zy1mZykiPkxhdGVuY3k6IH4xNS0zMCBjeWNsZXM8L3RleHQ+CiAgPHJlY3QgeD0iMzEw
IiB5PSIzMCIgd2lkdGg9IjIyMCIgaGVpZ2h0PSIxMjAiIHJ4PSI2IiBmaWxsPSJ2YXIoLS1zdmct
dGludC0xLCB2YXIoLS1zdmctdGludC0xLWZhbGxiYWNrKSkiIHN0cm9rZT0idmFyKC0tc3ZnLWZn
KSIvPgogIDx0ZXh0IHg9IjMyNSIgeT0iNTUiIGZvbnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJi
b2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5GdW5jdGlvbiBQb2ludGVyICgqZik8L3RleHQ+CiAg
PHRleHQgeD0iMzI1IiB5PSI4MCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+
4oCiIFJhdyA2NC1iaXQgYWRkcmVzczwvdGV4dD4KICA8dGV4dCB4PSIzMjUiIHk9Ijk4IiBmb250
LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj7igKIgWmVybyBjYXB0dXJlZCBzdGF0ZTwv
dGV4dD4KICA8dGV4dCB4PSIzMjUiIHk9IjExNiIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0t
c3ZnLWZnKSI+4oCiIEluZGlyZWN0IGp1bXAgaW5zdHJ1Y3Rpb248L3RleHQ+CiAgPHRleHQgeD0i
MzI1IiB5PSIxMzQiIGZvbnQtc2l6ZT0iMTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPkxhdGVuY3k6
IH41LTEwIGN5Y2xlczwvdGV4dD4KICA8cmVjdCB4PSI1NzAiIHk9IjMwIiB3aWR0aD0iMjMwIiBo
ZWlnaHQ9IjEyMCIgcng9IjYiIGZpbGw9InZhcigtLXN2Zy10aW50LTEsIHZhcigtLXN2Zy10aW50
LTEtZmFsbGJhY2spKSIgc3Ryb2tlPSJ2YXIoLS1zdmctZmcpIi8+CiAgPHRleHQgeD0iNTg1IiB5
PSI1NSIgZm9udC1zaXplPSIxNCIgZm9udC13ZWlnaHQ9ImJvbGQiIGZpbGw9InZhcigtLXN2Zy1m
ZykiPlRlbXBsYXRlZCBGdW5jdG9yIChhdXRvKTwvdGV4dD4KICA8dGV4dCB4PSI1ODUiIHk9Ijgw
IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj7igKIgQ29uY3JldGUgY29tcGls
ZXIgdHlwZTwvdGV4dD4KICA8dGV4dCB4PSI1ODUiIHk9Ijk4IiBmb250LXNpemU9IjE0IiBmaWxs
PSJ2YXIoLS1zdmctZmcpIj7igKIgQ2FwdHVyZXMgYXJiaXRyYXJ5IHN0YXRlPC90ZXh0PgogIDx0
ZXh0IHg9IjU4NSIgeT0iMTE2IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj7i
gKIgRnVsbHkgaW5saW5lZCBieSBjb21waWxlcjwvdGV4dD4KICA8dGV4dCB4PSI1ODUiIHk9IjEz
NCIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+TGF0ZW5jeTogMCBjeWNsZXMg
KElubGluZWQhKTwvdGV4dD4KPC9zdmc+
" alt="Figure 4: Callable Types in Modern C++" />
<figcaption>Figure 4: Callable Types in Modern C++</figcaption>
</figure>

**Specification:**

Implement a benchmarking harness that applies a mathematical
transformation ($f(x) = x \times 1.05 + 0.02$) across a vector of
20,000,000 `double` values using:

1.  A raw function pointer:
    `void transform_fp(std::span<double> data, double (*fn)(double))`.
2.  A type-erased wrapper:
    `void transform_std(std::span<double> data, const std::function<double(double)>& fn)`.
3.  A templated functor:
    `template <typename F> void transform_template(std::span<double> data, F fn)`.
    Measure the execution time of each and explain the architectural
    causes of the performance differences.

``` cpp
#include <span>
#include <functional>

void transform_fp(std::span<double> data, double (*fn)(double)) {
    for (auto& x : data) x = fn(x);
}

void transform_std(std::span<double> data, const std::function<double(double)>& fn) {
    for (auto& x : data) x = fn(x);
}

template <typename F>
void transform_template(std::span<double> data, F fn) {
    for (auto& x : data) x = fn(x);
}
```

**Verification Tasks:**

- **Task 1 (Correctness Check):** Verify that all three functions mutate
  an array of ones (`1.0`) into `1.07` accurately.
- **Task 2 (Throughput Profiling):** Profile all three on 20,000,000
  elements. Verify that the templated functor approach runs
  substantially faster than `std::function` (typically $4\times$ to
  $10\times$ faster).
- **Task 3 (SBO Heap Allocation Inspection):** Construct a lambda that
  captures an array of 64 bytes (`std::array<char, 64>`). Verify with
  global `operator new` tracking (from Chapter 10) that assigning this
  closure to `std::function` triggers a heap allocation, whereas passing
  it to `transform_template` triggers zero allocations.

#### Exercise 11.4: Monadic Pipeline Composition with `std::optional` (C++23)

**Context and Theory:**

In procedural code, operations that can fail are typically structured
with deeply nested conditional checks or error codes:

``` cpp
// Procedural Error Handling: The Pyramid of Doom
User* user = find_user(id);
if (user) {
    Session* session = user->get_session();
    if (session) {
        Token* token = session->get_token();
        if (token) { /* use token */ }
    }
}
```

This pattern—often called the *Pyramid of Doom*—obscures core business
logic with defensive error handling.

In functional programming, computations that might return nothing are
modelled using the **Maybe / Option Monad**. Modern C++ (C++23)
introduces **Monadic Operations** for `std::optional`:

- **`.and_then(F)`:** Takes a callable that returns an `optional<U>`. If
  the current instance contains a value, it executes `F` with that
  value; otherwise, it passes `std::nullopt` through without calling
  `F`.
- **`.transform(F)`:** Takes a callable that returns a plain value `U`.
  If a value is present, it wraps the transformed result into
  `optional<U>`.
- **`.or_else(F)`:** Invoked only if the current instance is empty,
  allowing the specification of a fallback computation.

This enables **Railway-Oriented Programming**, where computation flows
through a linear pipeline of pure transformations, short-circuiting to
an error state automatically upon the first failure.

**Specification:**

Design a user authentication pipeline using C++23 monadic operations:

1.  Define records: `UserRecord`, `SessionRecord`, and `AuthToken`.
2.  Implement three independent pure functions:
    - `fetch_user(int user_id) -> std::optional<UserRecord>`
    - `extract_session(const UserRecord& user) -> std::optional<SessionRecord>`
    - `validate_token(const SessionRecord& session) -> std::optional<AuthToken>`
3.  Chain these operations into a single functional pipeline without
    using any `if` statements.
4.  Use `.or_else()` to log when authentication fails and return a
    default guest token.

``` cpp
#include <optional>
#include <string>

struct UserRecord { int id; bool is_banned; std::string session_token; };
struct SessionRecord { std::string token; bool is_expired; };
struct AuthToken { std::string signed_hash; };

[[nodiscard]] std::optional<UserRecord> fetch_user(int id);
[[nodiscard]] std::optional<SessionRecord> extract_session(const UserRecord& u);
[[nodiscard]] std::optional<AuthToken> validate_token(const SessionRecord& s);

[[nodiscard]] std::optional<AuthToken> authenticate(int user_id) {
    return fetch_user(user_id)
        .and_then(extract_session)
        .and_then(validate_token);
}
```

**Verification Tasks:**

- **Task 1 (Happy Path Authentication):** Provide an active user ID with
  a valid session and unexpired token. Verify that `authenticate(id)`
  returns an `AuthToken` containing the valid signed hash.
- **Task 2 (Short-Circuit on Banned User):** Provide a user ID marked
  `is_banned = true`. Ensure `fetch_user` returns `std::nullopt`. Verify
  that execution terminates immediately without calling
  `extract_session` or `validate_token`.
- **Task 3 (Fallback Generation via `.or_else()`):** Append an
  `.or_else()` fallback to the pipeline. Pass an invalid user ID. Verify
  that the pipeline returns a guest authentication token rather than an
  empty `std::nullopt`.

#### Exercise 11.5: The Strategy Pattern: Classical OOP vs Closed Sum-Type (`std::variant`)

**Context and Theory:**

The **Strategy Pattern** decouples an algorithmic policy from the
context class that uses it.

- **Classical OOP Approach (Open Polymorphism):** An abstract base class
  `IDiscountStrategy` defines an interface. Concrete subclasses
  (`VipDiscount`, `SeasonalDiscount`) implement this interface. The
  client stores a `std::unique_ptr<IDiscountStrategy>`. This
  architecture is *open*: new strategies can be added across compilation
  units without recompiling the client. However, every strategy requires
  a heap allocation and dynamic virtual dispatch.
- **Modern Functional Approach (Closed Polymorphism):** The set of
  permitted strategies is bounded and expressed as a **Sum Type**:
  `using DiscountStrategy = std::variant<VipDiscount, SeasonalDiscount, FlatDiscount>;`
  The client stores this variant **directly by value** without heap
  allocation. Operations are dispatched using `std::visit`. The compiler
  constructs a tight jump table or inline conditional branch, completely
  eliminating dynamic dispatch overhead.

**Specification:**

Implement an e-commerce order pricing engine in both styles:

1.  **OOP Engine:** Abstract base `IDiscount`, subclasses `NoDiscount`,
    `PercentageDiscount(rate)`, and `VipDiscount(tier)`.
2.  **Variant Engine:** Value structs `NoDiscount`,
    `PercentageDiscount`, and `VipDiscount` bundled in a `std::variant`.
3.  Write an order calculator for both: given an order total of £100,
    compute the discounted price.
4.  Compare memory footprints (`sizeof`) and performance over a
    collection of 5,000,000 orders.

``` cpp
#include <variant>

struct NoDiscount {};
struct PercentageDiscount { double percent; };
struct VipDiscount { int tier; };

using StrategyVariant = std::variant<NoDiscount, PercentageDiscount, VipDiscount>;

[[nodiscard]] constexpr double apply_discount(double total, const StrategyVariant& strat) noexcept {
    return std::visit([total](const auto& s) noexcept -> double {
        using T = std::decay_t<decltype(s)>;
        if constexpr (std::is_same_v<T, PercentageDiscount>) {
            return total * (1.0 - s.percent);
        } else if constexpr (std::is_same_v<T, VipDiscount>) {
            return total - (s.tier * 10.0);
        }
        return total;
    }, strat);
}
```

**Verification Tasks:**

- **Task 1 (Pricing Equivalence):** Evaluate an order of £200 with a 15%
  discount and a Tier-2 VIP discount across both implementations. Verify
  that both yield £170 and £180 respectively.
- **Task 2 (Memory Footprint Inspection):** Verify that the variant
  strategy requires zero dynamic heap allocations, whereas the OOP
  approach requires a separate heap allocation for every unique strategy
  instance.
- **Task 3 (Exhaustive Pattern Matching):** Add a new struct
  `HolidaySpecialDiscount` to the variant. Verify that the compiler
  raises an error if you fail to handle it inside `std::visit`.

#### Exercise 11.6: Persistent Immutable Stack with Structural Sharing

**Context and Theory:**

In imperative programming, data structures are **ephemeral**: modifying
a collection overwrites its memory in place, destroying previous
versions of its state.

In pure functional programming, data structures are **persistent**:
performing an operation (such as `push` or `pop`) does not mutate the
existing structure; instead, it returns a brand-new structure
representing the updated state, while the old version remains intact and
accessible.

To make persistent data structures practical, they rely on **Structural
Sharing**. When a new element is pushed onto a persistent singly-linked
stack, the new stack node simply points to the head node of the previous
stack. Both versions share the exact same tail nodes in physical memory
(see fig. 5):

<figure id="fig:structural-sharing">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI1NzkuNjcyIiBo
ZWlnaHQ9Ijg3LjgwOCIgY2xhc3M9InN2Z2JvYiIgdmlld0JveD0iMCAtNCA1NzkuNjcyIDg3Ljgw
OCI+CiAgPHN0eWxlPi5zdmdib2IgbGluZSwgLnN2Z2JvYiBwYXRoLCAuc3ZnYm9iIGNpcmNsZSwg
LnN2Z2JvYiByZWN0LCAuc3ZnYm9iIHBvbHlnb24gewogIHN0cm9rZTogYmxhY2s7CiAgc3Ryb2tl
LXdpZHRoOiAxLjU7CiAgc3Ryb2tlLW9wYWNpdHk6IDE7CiAgZmlsbC1vcGFjaXR5OiAxOwogIHN0
cm9rZS1saW5lY2FwOiByb3VuZDsKICBzdHJva2UtbGluZWpvaW46IG1pdGVyOwp9Cgouc3ZnYm9i
IHRleHQgewogIHdoaXRlLXNwYWNlOiBwcmU7CiAgZmlsbDogYmxhY2s7CiAgZm9udC1mYW1pbHk6
IElvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtLCBtb25vc3BhY2U7CiAgZm9udC1zaXplOiAxNHB4
Owp9Cgouc3ZnYm9iIHJlY3QuYmFja2Ryb3AgewogIHN0cm9rZTogbm9uZTsKICBmaWxsOiB0cmFu
c3BhcmVudDsKfQoKLnN2Z2JvYiAuYnJva2VuIHsKICBzdHJva2UtZGFzaGFycmF5OiA4Owp9Cgou
c3ZnYm9iIC5maWxsZWQgewogIGZpbGw6IGJsYWNrOwp9Cgouc3ZnYm9iIC5iZ19maWxsZWQgewog
IGZpbGw6IHRyYW5zcGFyZW50OwogIHN0cm9rZS13aWR0aDogMTsKfQoKLnN2Z2JvYiAubm9maWxs
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9hcnJvdyB7CiAg
bWFya2VyLWVuZDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9hcnJvdyB7
CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Fycm93KTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9kaWFt
b25kIHsKICBtYXJrZXItZW5kOiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJr
ZWRfZGlhbW9uZCB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2RpYW1vbmQpOwp9Cgouc3ZnYm9iIC5l
bmRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNjaXJjbGUpOwp9Cgouc3ZnYm9i
IC5zdGFydF9tYXJrZWRfY2lyY2xlIHsKICBtYXJrZXItc3RhcnQ6IHVybCgjY2lyY2xlKTsKfQoK
LnN2Z2JvYiAuZW5kX21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2VyLWVuZDogdXJsKCNvcGVu
X2NpcmNsZSk7Cn0KCi5zdmdib2IgLnN0YXJ0X21hcmtlZF9vcGVuX2NpcmNsZSB7CiAgbWFya2Vy
LXN0YXJ0OiB1cmwoI29wZW5fY2lyY2xlKTsKfQoKLnN2Z2JvYiAuZW5kX21hcmtlZF9iaWdfb3Bl
bl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjYmlnX29wZW5fY2lyY2xlKTsKfQoKLnN2Z2Jv
YiAuc3RhcnRfbWFya2VkX2JpZ19vcGVuX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2Jp
Z19vcGVuX2NpcmNsZSk7Cn0KCjwvc3R5bGU+CiAgPGRlZnM+CiAgICA8bWFya2VyIGlkPSJhcnJv
dyIgdmlld0JveD0iLTIgLTIgOCA4IiByZWZYPSI0IiByZWZZPSIyIiBtYXJrZXJXaWR0aD0iNyIg
bWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxwb2x5
Z29uIHBvaW50cz0iMCwwIDAsNCA0LDIgMCwwIj48L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAg
IDxtYXJrZXIgaWQ9ImRpYW1vbmQiIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0i
MiIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJl
dmVyc2UiPgogICAgICA8cG9seWdvbiBwb2ludHM9IjAsMiAyLDAgNCwyIDIsNCAwLDIiPjwvcG9s
eWdvbj4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iY2lyY2xlIiB2aWV3Qm94PSIwIDAg
OCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBv
cmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiBy
PSIyIiBjbGFzcz0iZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBp
ZD0ib3Blbl9jaXJjbGUiIHZpZXdCb3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtl
cldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4K
ICAgICAgPGNpcmNsZSBjeD0iNCIgY3k9IjQiIHI9IjIiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2ly
Y2xlPgogICAgPC9tYXJrZXI+CiAgICA8bWFya2VyIGlkPSJiaWdfb3Blbl9jaXJjbGUiIHZpZXdC
b3g9IjAgMCA4IDgiIHJlZlg9IjQiIHJlZlk9IjQiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWln
aHQ9IjciIG9yaWVudD0iYXV0by1zdGFydC1yZXZlcnNlIj4KICAgICAgPGNpcmNsZSBjeD0iNCIg
Y3k9IjQiIHI9IjMiIGNsYXNzPSJiZ19maWxsZWQiPjwvY2lyY2xlPgogICAgPC9tYXJrZXI+CiAg
PC9kZWZzPgogIDxyZWN0IGNsYXNzPSJiYWNrZHJvcCIgeD0iMCIgeT0iLTQiIHdpZHRoPSI1Nzku
NjcyIiBoZWlnaHQ9Ijg3LjgwOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUi
IHg9IjI4OC4wOSIgeT0iMTAuNDc2IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIzMDIuMDU4IiB5PSIxMC40NzYiID5Ob2RlPC90ZXh0PgogIDxwb2x5Z29uIHBvaW50
cz0iMzIxLjI2NCwyNC40NDQgMzI0Ljc1NiwxMy45NjggMzI4LjI0ODAyLDI0LjQ0NCIgY2xhc3M9
ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMzNi45
NzgiIHk9IjEwLjQ3NiIgPjEwPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjM1Ny45MzAwMiIgeT0iMTAuNDc2IiA+XTwvdGV4dD4KICA8bGluZSB4MT0iMzcwLjE1MiIgeTE9
IjYuOTg0IiB4Mj0iMzg0LjEyMDAzIiB5Mj0iNi45ODQiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgog
IDxwb2x5Z29uIHBvaW50cz0iMzg0LjEyMDAzLDMuNDkyIDM5MS4xMDQsNi45ODQgMzg0LjEyMDAz
LDEwLjQ3NiIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjM5OS44MzQiIHk9IjEwLjQ3NiIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNl
PSJwcmVzZXJ2ZSIgeD0iNDEzLjgwMiIgeT0iMTAuNDc2IiA+Tm9kZTwvdGV4dD4KICA8dGV4dCB4
bWw6c3BhY2U9InByZXNlcnZlIiB4PSI0NDguNzIyMDIiIHk9IjEwLjQ3NiIgPjIwPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjQ2OS42NzQiIHk9IjEwLjQ3NiIgPl08L3Rl
eHQ+CiAgPGxpbmUgeDE9IjQ4MS44OTYwMyIgeTE9IjYuOTg0IiB4Mj0iNDk1Ljg2NCIgeTI9IjYu
OTg0IiBjbGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9IjQ5NS44NjQsMy40
OTIgNTAyLjg0ODAyLDYuOTg0IDQ5NS44NjQsMTAuNDc2IiBjbGFzcz0iZmlsbGVkIj48L3BvbHln
b24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTExLjU3OCIgeT0iMTAuNDc2IiA+
bnVsbHB0cjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjc0NiIgeT0i
MjQuNDQ0IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNS43MTQw
MDEiIHk9IjI0LjQ0NCIgPjEwLDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSI0My42NSIgeT0iMjQuNDQ0IiA+MjA8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iNjQuNjAyMDA1IiB5PSIyNC40NDQiID5dPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjE5Ny4yOTgiIHk9IjUyLjM4IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3Bh
Y2U9InByZXNlcnZlIiB4PSIyMTEuMjY2IiB5PSI1Mi4zOCIgPk5vZGU8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMjQ2LjE4NiIgeT0iNTIuMzgiID45OTwvdGV4dD4KICA8
dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyNjcuMTM4IiB5PSI1Mi4zOCIgPl08L3RleHQ+
CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS43NDYiIHk9IjY2LjM0OCIgPls8L3Rl
eHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTUuNzE0MDAxIiB5PSI2Ni4zNDgi
ID45OSw8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDMuNjUiIHk9IjY2
LjM0OCIgPjEwLDwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSI3MS41ODYi
IHk9IjY2LjM0OCIgPjIwPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9Ijky
LjUzOCIgeT0iNjYuMzQ4IiA+XTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4
PSIxLjc0NiIgeT0iMTAuNDc2IiA+VmVyc2lvbiAxIChJbml0aWFsIFN0YWNrKTo8L3RleHQ+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMS43NDYiIHk9IjUyLjM4IiA+VmVyc2lvbiAy
IChBZnRlciBwdXNoIDk5KTo8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0i
MzQzLjk2MiIgeT0iNTIuMzgiID4oWmVybyBjb3B5aW5nIG9mIE5vZGVzIDEwIG9yIDIwISk8L3Rl
eHQ+CiAgPGc+CiAgICA8bGluZSB4MT0iMzI0Ljc1NiIgeTE9IjI0LjQ0NCIgeDI9IjMyNC43NTYi
IHkyPSI0OC44ODgiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogICAgPGxpbmUgeDE9IjI3OS4zNjAw
MiIgeTE9IjQ4Ljg4OCIgeDI9IjMyNC43NTYiIHkyPSI0OC44ODgiIGNsYXNzPSJzb2xpZCI+PC9s
aW5lPgogIDwvZz4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 5: Structural Sharing</figcaption>
</figure>

Structural sharing provides:

- **Zero-Copy Persistence:** Creating a new version runs in $O(1)$ time
  and allocates only one node.
- **Intrinsic Thread Safety:** Because nodes are completely immutable,
  multiple threads can read, traverse, and branch off the same
  underlying nodes simultaneously without synchronization primitives or
  mutex locks.

**Specification:**

Implement `PersistentStack<T>`:

- Internal nodes must be immutable:
  `struct Node { const T data; const std::shared_ptr<const Node> next; };`.
- The stack holds only a single member:
  `std::shared_ptr<const Node> m_head;`.
- Implement `push(T value) const -> PersistentStack<T>`: returns a new
  stack sharing the existing tail.
- Implement `pop() const -> PersistentStack<T>`: returns a new stack
  referencing `m_head->next`.
- Implement `top() const -> const T&`.
- Verify that older versions remain unaltered after pushes and pops.

``` cpp
#include <memory>
#include <stdexcept>

template <typename T>
class PersistentStack {
    struct Node {
        T data;
        std::shared_ptr<const Node> next;
    };
    std::shared_ptr<const Node> m_head{nullptr};
    explicit PersistentStack(std::shared_ptr<const Node> head) : m_head(std::move(head)) {}
public:
    PersistentStack() = default;

    [[nodiscard]] PersistentStack push(T value) const {
        return PersistentStack(std::make_shared<const Node>(std::move(value), m_head));
    }
    [[nodiscard]] PersistentStack pop() const {
        if (!m_head) throw std::underflow_error("Stack is empty");
        return PersistentStack(m_head->next);
    }
    [[nodiscard]] const T& top() const {
        if (!m_head) throw std::underflow_error("Stack is empty");
        return m_head->data;
    }
    [[nodiscard]] bool empty() const noexcept { return m_head == nullptr; }
};
```

**Verification Tasks:**

- **Task 1 (Immutability and Version Independence):** Create `v1`. Push
  `10` and `20`. From `v1`, create `v2 = v1.push(30)` and
  `v3 = v1.push(40)`. Verify that `v1.top() == 20`, `v2.top() == 30`,
  and `v3.top() == 40`.
- **Task 2 (Structural Memory Sharing):** Verify that the address of
  `v2.pop().top()` is identical to the address of `v1.top()`, confirming
  that internal nodes are shared rather than copied.
- **Task 3 (Multi-Threaded Safe Concurrent Traversal):** Pass `v1` to
  four concurrent threads. Allow each thread to push 10,000 distinct
  items without mutex locks. Verify that all operations complete without
  data races and that `v1` remains unaltered.

#### Exercise 11.7: Mechanical Sympathy and SIMD Auto-Vectorisation

**Context and Theory:**

Modern processors achieve high throughput using **Single Instruction,
Multiple Data (SIMD)** vector execution units (such as Intel AVX-512 or
ARM NEON). A 256-bit AVX vector register can execute eight 32-bit
floating-point multiplications in a single clock cycle.

However, compilers can only auto-vectorise code when the implementation
exhibits **Mechanical Sympathy** with the underlying hardware:

1.  **Contiguous Memory:** Data must reside sequentially in memory
    without pointer indirection (avoiding arrays of pointers).
2.  **Absence of Pointer Aliasing:** If two pointer arguments might
    point to overlapping memory regions, the compiler cannot reorder or
    vectorise memory writes. In C++, developers inform the compiler of
    non-overlapping buffers using the non-standard `__restrict__`
    keyword or by using value-semantic containers.
3.  **Branchless Inner Loops:** Eliminating data-dependent `if-else`
    branches inside tight loops prevents pipeline stalls and allows
    operations to map directly to masked vector instructions.

**Specification:**

Design an experiment to analyse the compiler’s auto-vectorisation
capabilities:

1.  Write a procedural loop that computes the element-wise sum and clamp
    of two arrays:
    $$Z[i] = \min(\max(X[i] + Y[i], 0.0\text{f}), 1.0\text{f})$$
2.  Compile with `-O3 -fopt-info-vec-optimized` (on GCC/Clang) and
    examine the vectorisation diagnostics.
3.  Compare the assembly generated when pointers are unconstrained
    versus when pointers are qualified with `__restrict__`.
4.  Measure the throughput delta between the scalar unvectorised version
    and the SIMD vectorised version.

``` cpp
#include <cstddef>
#include <algorithm>

void compute_clamped_sum(
    const float* __restrict__ x,
    const float* __restrict__ y,
    float* __restrict__ z,
    std::size_t count) noexcept 
{
    for (std::size_t i = 0; i < count; ++i) {
        z[i] = std::clamp(x[i] + y[i], 0.0f, 1.0f);
    }
}
```

**Verification Tasks:**

- **Task 1 (Compiler Diagnostic Verification):** Compile the function
  using GCC/Clang with `-O3 -mavx2 -fopt-info-vec-all`. Verify that the
  compiler outputs: `loop vectorized using 32-byte vectors`.
- **Task 2 (Numerical Clamping Correctness):** Pass input arrays
  $X = [-0.5, 0.4, 0.8]$ and $Y = [0.2, 0.3, 0.9]$. Verify that the
  computed array $Z$ equals $[0.0, 0.7, 1.0]$.
- **Task 3 (Aliasing Performance Delta):** Remove the `__restrict__`
  qualifiers and run both versions on 50,000,000 elements. Verify that
  the vectorised version achieves a $4\times$ to $8\times$ performance
  increase over the aliased version.

#### Exercise 11.8: Currying and Partial Application via Higher-Order Closures

**Context and Theory:**

In programming language theory, **Currying** (named after logician
Haskell Curry) is the technique of translating the evaluation of a
function that takes multiple arguments:

$$f: (A \times B \times C) \to D$$

into a sequence of unary functions that each take a single argument:

$$f': A \to (B \to (C \to D))$$

**Partial Application** is a closely related technique where a subset of
a function’s arguments are pre-bound, producing a new function of lower
arity:

$$\text{bind}_{1}(f, a) = g(b, c) = f(a, b, c)$$

In Modern C++, we implement currying and partial application using
**Higher-Order Generic Lambdas** that capture state by value. This
allows configuration data (such as network timeouts, database handles,
or scaling factors) to be bound early, generating specialised functions
that slot cleanly into generic algorithm pipelines (`std::transform`,
`std::ranges`).

**Specification:**

Implement generic currying utilities in C++20:

1.  Write a higher-order factory
    `make_curried_multiplier(double factor)`: returns a unary lambda
    that multiplies its argument by `factor`.
2.  Write a general-purpose curry wrapper `curry(F f)` that takes a
    binary function $f(a, b)$ and transforms it into a curried callable
    `f(a)(b)`.
3.  Construct a sensor calibration pipeline: curry a three-parameter
    calibration equation:
    $$\text{Calibrated}(s, o, v) = (v + o) \times s$$ Bind the scale $s$
    and offset $o$ up front, producing a single unary function
    `calibrator(v)` suitable for passing directly into standard range
    transformations.

``` cpp
#include <concepts>
#include <utility>

// Generic binary currying wrapper
template <typename F>
[[nodiscard]] constexpr auto curry2(F f) {
    return [f](auto a) {
        return [f, a](auto b) {
            return f(a, b);
        };
    };
}
```

**Verification Tasks:**

- **Task 1 (Binary Currying Evaluation):** Define a binary lambda
  `auto add = [](int a, int b) { return a + b; };`. Instantiate
  `auto curried_add = curry2(add);`. Verify that
  `curried_add(5)(7) == 12`.
- **Task 2 (Partial Application Pipeline):** Create an adder
  `auto add10 = curried_add(10);`. Pass `add10` to `std::transform` over
  an array `[1, 2, 3]`. Verify the output vector is `[11, 12, 13]`.
- **Task 3 (Ternary Calibrator):** Curry the calibration formula
  $(v + 2.5) \times 1.8$. Evaluate raw voltage $v = 10.0$. Verify that
  the result computes $(10.0 + 2.5) \times 1.8 = 22.5$ accurately.

#### Exercise 11.9: Static Polymorphism via the Curiously Recurring Template Pattern (CRTP)

**Context and Theory:**

Dynamic polymorphism via virtual functions uses runtime dispatch through
a `vtable`, which incurs pointer indirection and inhibits function
inlining.

When polymorphic relationships are known entirely at compile time,
Modern C++ uses **Static Polymorphism** via the **Curiously Recurring
Template Pattern (CRTP)**.

In CRTP, a base class is templated on its derived class:

``` cpp
template <typename Derived>
class Base { ... };

class Concrete : public Base<Concrete> { ... };
```

The base class accesses the derived class by downcasting its `this`
pointer statically (see fig. 6):

$$\text{static\_cast}\langle\text{Derived}*\rangle(\text{this})$$

- Dynamic dispatch is completely bypassed: the compiler resolves
  function calls at compile time.
- The method body is fully exposed to the compiler for direct inlining.
- Memory footprint is reduced: CRTP instances contain no hidden
  `__vptr`, saving 8 bytes per object.

<figure id="fig:static-polymorphism">
<img src="data:image/svg+xml;base64,PHN2ZyB2aWV3Qm94PSI0MCAyMCA3NzAgMTIwIiB3aWR0aD0iMTAwJSIgaGVpZ2h0PSIxMjAiIHht
bG5zPSJodHRwOi8vd3d3LnczLm9yZy8yMDAwL3N2ZyI+CiAgPGRlZnM+CiAgICA8c3R5bGU+CiAg
ICAgIDpyb290IHsKICAgICAgICAtLXN2Zy1iZzogdmFyKC0tYmcsICNmZmZmZmYpOwogICAgICAg
IC0tc3ZnLWZnOiB2YXIoLS1mZywgIzAwMDAwMCk7CiAgICAgICAgLS1zdmctdGludC0xLWZhbGxi
YWNrOiAjY2NjY2NjOyAvKiAyMCUgYmxhY2sgKyA4MCUgd2hpdGUgKi8KICAgICAgICAtLXN2Zy10
aW50LTItZmFsbGJhY2s6ICM5OTk5OTk7IC8qIDQwJSBibGFjayArIDYwJSB3aGl0ZSAqLwogICAg
ICAgIC0tc3ZnLXRpbnQtMTogY29sb3ItbWl4KGluIHNyZ2IsIHZhcigtLXN2Zy1mZykgMjAlLCB2
YXIoLS1zdmctYmcpIDgwJSk7CiAgICAgICAgLS1zdmctdGludC0yOiBjb2xvci1taXgoaW4gc3Jn
YiwgdmFyKC0tc3ZnLWZnKSA0MCUsIHZhcigtLXN2Zy1iZykgNjAlKTsKICAgICAgfQogICAgICAu
YmcgICAgIHsgZmlsbDogdmFyKC0tc3ZnLWJnLCAjZmZmZmZmKTsgfQogICAgICAuZmcgICAgIHsg
ZmlsbDogdmFyKC0tc3ZnLWZnLCAjMDAwMDAwKTsgc3Ryb2tlOiB2YXIoLS1zdmctZmcsICMwMDAw
MDApOyB9CiAgICAgIC50aW50LTEgeyBmaWxsOiB2YXIoLS1zdmctdGludC0xLCB2YXIoLS1zdmct
dGludC0xLWZhbGxiYWNrKSk7IH0KICAgICAgLnRpbnQtMiB7IGZpbGw6IHZhcigtLXN2Zy10aW50
LTIsIHZhcigtLXN2Zy10aW50LTItZmFsbGJhY2spKTsgfQogICAgICB0ZXh0ICAgeyBmb250LWZh
bWlseTogIklvc2V2a2EgQ2hhcm9uIE1vbm8gTWVkaXVtIiwgbW9ub3NwYWNlOyB9CiAgICA8L3N0
eWxlPgogIDwvZGVmcz4KICA8cmVjdCB4PSI1MCIgeT0iMzAiIHdpZHRoPSIzNDAiIGhlaWdodD0i
MTAwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRpbnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxs
YmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZykiLz4KICA8dGV4dCB4PSI2NSIgeT0iNTUiIGZv
bnQtc2l6ZT0iMTQiIGZvbnQtd2VpZ2h0PSJib2xkIiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5DUlRQ
IEJhc2UgQ2xhc3MgVGVtcGxhdGU8L3RleHQ+CiAgPHRleHQgeD0iNjUiIHk9IjgwIiBmb250LXNp
emU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj50ZW1wbGF0ZSAmbHQ7dHlwZW5hbWUgRGVyaXZl
ZCZndDs8L3RleHQ+CiAgPHRleHQgeD0iNjUiIHk9Ijk4IiBmb250LXNpemU9IjE0IiBmaWxsPSJ2
YXIoLS1zdmctZmcpIj5jbGFzcyBOdW1lcmljU2Vuc29yIHs8L3RleHQ+CiAgPHRleHQgeD0iNzUi
IHk9IjExNiIgZm9udC1zaXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+c3RhdGljX2Nhc3Qm
bHQ7RGVyaXZlZComZ3Q7KHRoaXMpLSZndDtyZWFkKCk7PC90ZXh0PgogIDxyZWN0IHg9IjQ2MCIg
eT0iMzAiIHdpZHRoPSIzNDAiIGhlaWdodD0iMTAwIiByeD0iNiIgZmlsbD0idmFyKC0tc3ZnLXRp
bnQtMSwgdmFyKC0tc3ZnLXRpbnQtMS1mYWxsYmFjaykpIiBzdHJva2U9InZhcigtLXN2Zy1mZyki
Lz4KICA8dGV4dCB4PSI0NzUiIHk9IjU1IiBmb250LXNpemU9IjE0IiBmb250LXdlaWdodD0iYm9s
ZCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+Q29uY3JldGUgRGVyaXZlZCBDbGFzczwvdGV4dD4KICA8
dGV4dCB4PSI0NzUiIHk9IjgwIiBmb250LXNpemU9IjE0IiBmaWxsPSJ2YXIoLS1zdmctZmcpIj5j
bGFzcyBUaGVybWFsU2Vuc29yIDo8L3RleHQ+CiAgPHRleHQgeD0iNDk1IiB5PSI5OCIgZm9udC1z
aXplPSIxNCIgZmlsbD0idmFyKC0tc3ZnLWZnKSI+cHVibGljIE51bWVyaWNTZW5zb3ImbHQ7VGhl
cm1hbFNlbnNvciZndDs8L3RleHQ+CiAgPHRleHQgeD0iNDc1IiB5PSIxMTgiIGZvbnQtc2l6ZT0i
MTQiIGZpbGw9InZhcigtLXN2Zy1mZykiPuKGkiBSZXNvbHZlZCBhdCBjb21waWxlIHRpbWUhIFpl
cm8gdnB0ciE8L3RleHQ+Cjwvc3ZnPg==
" alt="Figure 6: Implementation of the CRTP Pattern" />
<figcaption>Figure 6: Implementation of the CRTP Pattern</figcaption>
</figure>

**Specification:**

Design a statically polymorphic sensor hierarchy using CRTP:

1.  Create `SensorBase<Derived>` with a public method
    `double read_value() const` that calls
    `static_cast<const Derived*>(this)->read_implementation()`.
2.  Implement two derived classes: `TemperatureSensor` and
    `PressureSensor`.
3.  Write a generic processor template
    `template <typename S> void process_sensor(const SensorBase<S>& sensor)`.
4.  Assert using `sizeof` that instances of `TemperatureSensor` contain
    zero `vptr` overhead.

``` cpp
template <typename Derived>
class SensorBase {
public:
    [[nodiscard]] double read_value() const noexcept {
        return static_cast<const Derived*>(this)->read_implementation();
    }
};

class TemperatureSensor : public SensorBase<TemperatureSensor> {
public:
    [[nodiscard]] double read_implementation() const noexcept {
        return 21.5; // Direct compile-time binding!
    }
};
```

**Verification Tasks:**

- **Task 1 (Interface Invocation):** Instantiate `TemperatureSensor` and
  call `read_value()`. Verify it returns `21.5`.
- **Task 2 (Zero Memory Overhead Proof):** Assert via `static_assert`
  that `sizeof(TemperatureSensor) == 1` byte (empty struct size),
  whereas an equivalent class with a `virtual` method requires at least
  `8` bytes to store the `__vptr`.
- **Task 3 (Inlining Assembly Check):** Compile `process_sensor` on
  Compiler Explorer under `-O2`. Verify that the call is inlined
  directly, generating no function call instructions.

#### Exercise 11.10: Range Pipelines vs Intermediate Allocations (`std::ranges::views`)

**Context and Theory:**

In languages such as JavaScript or Python, functional data processing
chains incur severe performance penalties due to eager evaluation:

``` javascript
// JavaScript: Allocates THREE separate dynamic arrays on the heap!
let result = data.filter(x => x > 0).map(x => x * 2).slice(0, 10);
```

Each stage allocates a brand-new intermediate array, populates it, and
immediately discards it, placing heavy pressure on the garbage collector
and blowing out CPU caches.

Modern C++ (C++20) introduces **Ranges and Views (`std::views`)**.

A `view` is a **Lazy, Non-Owning, Composable Range Adaptor**. When you
compose views using the pipe operator (`|`):

- Zero intermediate heap allocations occur.
- Elements are evaluated **lazily on-demand** as the consumer iterates
  through the range.
- Operations are fused together: the filter, transform, and take stages
  execute in a single pass over the source data, keeping intermediate
  values resident inside CPU registers.

**Specification:**

Compare eager procedural allocation against lazy functional range views:

1.  Construct an array of 5,000,000 random integers.
2.  Pipeline: Filter even numbers, square them, and take the first 100
    elements.
3.  **Eager Implementation:** Uses standard `std::vector` allocations to
    hold intermediate filtered and mapped collections.
4.  **Range View Implementation:** Uses
    `data | std::views::filter(...) | std::views::transform(...) | std::views::take(100)`.
5.  Benchmark execution time and monitor total dynamic memory
    allocations using global operator overloads.

``` cpp
#include <ranges>
#include <vector>
#include <iostream>

auto process_ranges(const std::vector<int>& data) {
    auto is_even = [](int x) noexcept { return x % 2 == 0; };
    auto square  = [](int x) noexcept { return x * x; };

    // Lazy pipeline: 0 bytes allocated on the heap!
    return data 
        | std::views::filter(is_even)
        | std::views::transform(square)
        | std::views::take(100);
}
```

**Verification Tasks:**

- **Task 1 (Output Equivalence):** Verify that both the eager pipeline
  and lazy range pipeline produce the exact same sequence of 100
  integers.
- **Task 2 (Zero Allocation Verification):** Attach the memory leak and
  allocation profiler from Chapter 10. Verify that `process_ranges`
  executes with **0 heap allocations**, whereas the eager vector
  approach executes multiple dynamic allocations.
- **Task 3 (Lazy Evaluation Efficiency):** Provide a stream where only
  the first 200 elements need to be scanned to yield 100 evens. Confirm
  that the range pipeline terminates after scanning the first 200 items,
  while the eager approach processes all 5,000,000 items wastefully.

#### Exercise 11.11: Multi-Paradigm Image Convolution Filter

**Context and Theory:**

Image convolution applies a $3 \times 3$ kernel matrix $K$ across a 2D
image grid $I$, calculating each output pixel as the weighted sum of a
target pixel and its eight neighbours:

$$O(x, y) = \sum_{i=-1}^{1} \sum_{j=-1}^{1} I(x + i, y + j) \cdot K(i + 1, j + 1)$$

This domain highlights the practical trade-offs among paradigms:

- **Procedural:** Optimal for the inner convolution kernel where memory
  access patterns, pointer strides, and cache locality dictate
  processing speed.
- **Functional:** Ideal for declarative boundary management, coordinate
  translation, and kernel specification using immutable matrix records.
- **Object-Oriented:** Best suited for managing image resource
  lifecycles (RAII), file format encapsulation, and exposing a uniform
  `IFilter` interface.

**Specification:**

Design a multi-paradigm image processing architecture:

1.  **OOP Layer:** An `Image` class managing a contiguous dynamic buffer
    of 8-bit greyscale pixels (`std::unique_ptr<uint8_t[]>`),
    implementing RAII and exposing dimensions.
2.  **Functional Layer:** An immutable `Kernel3x3` struct holding
    normalized convolution weights (e.g., Gaussian Blur, Ridge
    Detection) configured using pure helper functions.
3.  **Procedural Layer:** An optimised inner execution loop
    `apply_convolution_kernel` using raw contiguous pointers and
    boundary clamping.

``` cpp
#include <memory>
#include <cstdint>
#include <array>

struct Kernel3x3 {
    std::array<float, 9> weights;
};

class Image {
    std::size_t m_width, m_height;
    std::unique_ptr<std::uint8_t[]> m_pixels;
public:
    Image(std::size_t w, std::size_t h) 
        : m_width(w), m_height(h), m_pixels(std::make_unique<std::uint8_t[]>(w * h)) {}

    [[nodiscard]] std::uint8_t& at(std::size_t x, std::size_t y) noexcept {
        return m_pixels[y * m_width + x];
    }
    [[nodiscard]] const std::uint8_t& at(std::size_t x, std::size_t y) const noexcept {
        return m_pixels[y * m_width + x];
    }
    [[nodiscard]] std::size_t width() const noexcept { return m_width; }
    [[nodiscard]] std::size_t height() const noexcept { return m_height; }
};
```

**Verification Tasks:**

- **Task 1 (Identity Kernel):** Apply an identity kernel (centre weight
  `1.0`, all others `0.0`) to a $100 \times 100$ image. Verify that the
  output image is identical to the input.
- **Task 2 (Box Blur Smoothing):** Apply a uniform box blur kernel
  ($\text{weight} = \frac{1}{9}$ for all elements) to an image
  containing a sharp white square on a black background. Verify that
  edges are smoothed to intermediate grey values
  ($0 < \text{pixel} < 255$).
- **Task 3 (RAII Memory Management):** Verify that constructing and
  discarding multiple high-resolution images ($4096 \times 4096$) cleans
  up all heap memory without leaks.

#### Exercise 11.12: Tail-Call Optimisation (TCO) and Recursion Transformation

**Context and Theory:**

In pure functional programming, iteration is expressed not through
mutable `for` loops, but through **Recursion**.

However, on physical Von Neumann architectures, recursive calls push a
new stack frame onto the call stack for every step. If a function
recurses 1,000,000 times, it will exhaust the stack (typically 8MB on
Linux), causing a **Stack Overflow** crash.

A **Tail Call** occurs when a function’s recursive invocation is the
**very last operation** performed before returning:

``` cpp
// NOT a Tail Call: Multiplication happens AFTER the call returns!
return n * factorial(n - 1);

// TAIL CALL: The call result is returned directly with NO further work!
return factorial_tail(n - 1, acc * n);
```

When a function is written in tail-recursive form, an optimizing
compiler can apply **Tail-Call Optimisation (TCO)**: it discards or
reuses the current stack frame instead of pushing a new one (see
fig. 7). This has the result of converting the recursion into a fast,
jump-based machine loop ($\text{Space Complexity: } O(N) \to O(1)$).

<figure id="fig:recursion-tco">
<img src="data:image/svg+xml;base64,PHN2ZyB4bWxucz0iaHR0cDovL3d3dy53My5vcmcvMjAwMC9zdmciIHdpZHRoPSI0NzQuOTEyMDIi
IGhlaWdodD0iODcuODA4IiBjbGFzcz0ic3ZnYm9iIiB2aWV3Qm94PSIwIC00IDQ3NC45MTIwMiA4
Ny44MDgiPgogIDxzdHlsZT4uc3ZnYm9iIGxpbmUsIC5zdmdib2IgcGF0aCwgLnN2Z2JvYiBjaXJj
bGUsIC5zdmdib2IgcmVjdCwgLnN2Z2JvYiBwb2x5Z29uIHsKICBzdHJva2U6IGJsYWNrOwogIHN0
cm9rZS13aWR0aDogMS41OwogIHN0cm9rZS1vcGFjaXR5OiAxOwogIGZpbGwtb3BhY2l0eTogMTsK
ICBzdHJva2UtbGluZWNhcDogcm91bmQ7CiAgc3Ryb2tlLWxpbmVqb2luOiBtaXRlcjsKfQoKLnN2
Z2JvYiB0ZXh0IHsKICB3aGl0ZS1zcGFjZTogcHJlOwogIGZpbGw6IGJsYWNrOwogIGZvbnQtZmFt
aWx5OiBJb3NldmthIENoYXJvbiBNb25vIE1lZGl1bSwgbW9ub3NwYWNlOwogIGZvbnQtc2l6ZTog
MTRweDsKfQoKLnN2Z2JvYiByZWN0LmJhY2tkcm9wIHsKICBzdHJva2U6IG5vbmU7CiAgZmlsbDog
dHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmJyb2tlbiB7CiAgc3Ryb2tlLWRhc2hhcnJheTogODsK
fQoKLnN2Z2JvYiAuZmlsbGVkIHsKICBmaWxsOiBibGFjazsKfQoKLnN2Z2JvYiAuYmdfZmlsbGVk
IHsKICBmaWxsOiB0cmFuc3BhcmVudDsKICBzdHJva2Utd2lkdGg6IDE7Cn0KCi5zdmdib2IgLm5v
ZmlsbCB7CiAgZmlsbDogdHJhbnNwYXJlbnQ7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYXJyb3cg
ewogIG1hcmtlci1lbmQ6IHVybCgjYXJyb3cpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfYXJy
b3cgewogIG1hcmtlci1zdGFydDogdXJsKCNhcnJvdyk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRf
ZGlhbW9uZCB7CiAgbWFya2VyLWVuZDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2JvYiAuc3RhcnRf
bWFya2VkX2RpYW1vbmQgewogIG1hcmtlci1zdGFydDogdXJsKCNkaWFtb25kKTsKfQoKLnN2Z2Jv
YiAuZW5kX21hcmtlZF9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgjY2lyY2xlKTsKfQoKLnN2
Z2JvYiAuc3RhcnRfbWFya2VkX2NpcmNsZSB7CiAgbWFya2VyLXN0YXJ0OiB1cmwoI2NpcmNsZSk7
Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1hcmtlci1lbmQ6IHVybCgj
b3Blbl9jaXJjbGUpOwp9Cgouc3ZnYm9iIC5zdGFydF9tYXJrZWRfb3Blbl9jaXJjbGUgewogIG1h
cmtlci1zdGFydDogdXJsKCNvcGVuX2NpcmNsZSk7Cn0KCi5zdmdib2IgLmVuZF9tYXJrZWRfYmln
X29wZW5fY2lyY2xlIHsKICBtYXJrZXItZW5kOiB1cmwoI2JpZ19vcGVuX2NpcmNsZSk7Cn0KCi5z
dmdib2IgLnN0YXJ0X21hcmtlZF9iaWdfb3Blbl9jaXJjbGUgewogIG1hcmtlci1zdGFydDogdXJs
KCNiaWdfb3Blbl9jaXJjbGUpOwp9Cgo8L3N0eWxlPgogIDxkZWZzPgogICAgPG1hcmtlciBpZD0i
YXJyb3ciIHZpZXdCb3g9Ii0yIC0yIDggOCIgcmVmWD0iNCIgcmVmWT0iMiIgbWFya2VyV2lkdGg9
IjciIG1hcmtlckhlaWdodD0iNyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8
cG9seWdvbiBwb2ludHM9IjAsMCAwLDQgNCwyIDAsMCI+PC9wb2x5Z29uPgogICAgPC9tYXJrZXI+
CiAgICA8bWFya2VyIGlkPSJkaWFtb25kIiB2aWV3Qm94PSItMiAtMiA4IDgiIHJlZlg9IjQiIHJl
Zlk9IjIiIG1hcmtlcldpZHRoPSI3IiBtYXJrZXJIZWlnaHQ9IjciIG9yaWVudD0iYXV0by1zdGFy
dC1yZXZlcnNlIj4KICAgICAgPHBvbHlnb24gcG9pbnRzPSIwLDIgMiwwIDQsMiAyLDQgMCwyIj48
L3BvbHlnb24+CiAgICA8L21hcmtlcj4KICAgIDxtYXJrZXIgaWQ9ImNpcmNsZSIgdmlld0JveD0i
MCAwIDggOCIgcmVmWD0iNCIgcmVmWT0iNCIgbWFya2VyV2lkdGg9IjciIG1hcmtlckhlaWdodD0i
NyIgb3JpZW50PSJhdXRvLXN0YXJ0LXJldmVyc2UiPgogICAgICA8Y2lyY2xlIGN4PSI0IiBjeT0i
NCIgcj0iMiIgY2xhc3M9ImZpbGxlZCI+PC9jaXJjbGU+CiAgICA8L21hcmtlcj4KICAgIDxtYXJr
ZXIgaWQ9Im9wZW5fY2lyY2xlIiB2aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBt
YXJrZXJXaWR0aD0iNyIgbWFya2VySGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJz
ZSI+CiAgICAgIDxjaXJjbGUgY3g9IjQiIGN5PSI0IiByPSIyIiBjbGFzcz0iYmdfZmlsbGVkIj48
L2NpcmNsZT4KICAgIDwvbWFya2VyPgogICAgPG1hcmtlciBpZD0iYmlnX29wZW5fY2lyY2xlIiB2
aWV3Qm94PSIwIDAgOCA4IiByZWZYPSI0IiByZWZZPSI0IiBtYXJrZXJXaWR0aD0iNyIgbWFya2Vy
SGVpZ2h0PSI3IiBvcmllbnQ9ImF1dG8tc3RhcnQtcmV2ZXJzZSI+CiAgICAgIDxjaXJjbGUgY3g9
IjQiIGN5PSI0IiByPSIzIiBjbGFzcz0iYmdfZmlsbGVkIj48L2NpcmNsZT4KICAgIDwvbWFya2Vy
PgogIDwvZGVmcz4KICA8cmVjdCBjbGFzcz0iYmFja2Ryb3AiIHg9IjAiIHk9Ii00IiB3aWR0aD0i
NDc0LjkxMjAyIiBoZWlnaHQ9Ijg3LjgwOCI+PC9yZWN0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJl
c2VydmUiIHg9IjEuNzQ2IiB5PSIyNC40NDQiID5bPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0i
cHJlc2VydmUiIHg9IjE1LjcxNDAwMSIgeT0iMjQuNDQ0IiA+RnJhbWU8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNTcuNjE4IiB5PSIyNC40NDQiID4xPC90ZXh0PgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjcxLjU4NiIgeT0iMjQuNDQ0IiA+XTwvdGV4dD4K
ICA8bGluZSB4MT0iODMuODA4IiB5MT0iMjAuOTUyIiB4Mj0iOTcuNzc2IiB5Mj0iMjAuOTUyIiBj
bGFzcz0ic29saWQiPjwvbGluZT4KICA8cG9seWdvbiBwb2ludHM9Ijk3Ljc3NiwxNy40NjAwMDEg
MTA0Ljc2LDIwLjk1MiA5Ny43NzYsMjQuNDQ0IiBjbGFzcz0iZmlsbGVkIj48L3BvbHlnb24+CiAg
PHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTEzLjQ5MDAwNSIgeT0iMjQuNDQ0IiA+Wzwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxMjcuNDU4IiB5PSIyNC40NDQi
ID5GcmFtZTwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxNjkuMzYyIiB5
PSIyNC40NDQiID4yPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjE4My4z
MyIgeT0iMjQuNDQ0IiA+XTwvdGV4dD4KICA8bGluZSB4MT0iMTk1LjU1MiIgeTE9IjIwLjk1MiIg
eDI9IjIwOS41MiIgeTI9IjIwLjk1MiIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHBvbHlnb24g
cG9pbnRzPSIyMDkuNTIsMTcuNDYwMDAxIDIxNi41MDQwMSwyMC45NTIgMjA5LjUyLDI0LjQ0NCIg
Y2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9
IjIyNS4yMzQwMSIgeT0iMjQuNDQ0IiA+WzwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNl
cnZlIiB4PSIyMzkuMjAyMDEiIHk9IjI0LjQ0NCIgPkZyYW1lPC90ZXh0PgogIDx0ZXh0IHhtbDpz
cGFjZT0icHJlc2VydmUiIHg9IjI4MS4xMDYwMiIgeT0iMjQuNDQ0IiA+MzwvdGV4dD4KICA8dGV4
dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyOTUuMDc0IiB5PSIyNC40NDQiID5dPC90ZXh0Pgog
IDx0ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjMwOS4wNDIwMiIgeT0iMjQuNDQ0IiA+Li4u
PC90ZXh0PgogIDxsaW5lIHgxPSIzMzUuMjMyIiB5MT0iMjAuOTUyIiB4Mj0iMzQ5LjIiIHkyPSIy
MC45NTIiIGNsYXNzPSJzb2xpZCI+PC9saW5lPgogIDxwb2x5Z29uIHBvaW50cz0iMzQ5LjIsMTcu
NDYwMDAxIDM1Ni4xODQwMiwyMC45NTIgMzQ5LjIsMjQuNDQ0IiBjbGFzcz0iZmlsbGVkIj48L3Bv
bHlnb24+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMzY0LjkxNCIgeT0iMjQuNDQ0
IiA+U1RBQ0s8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNDA2LjgxODAy
IiB5PSIyNC40NDQiID5PVkVSRkxPVyE8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2
ZSIgeD0iMS43NDYiIHk9IjY2LjM0OCIgPls8L3RleHQ+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVz
ZXJ2ZSIgeD0iMTUuNzE0MDAxIiB5PSI2Ni4zNDgiID5GcmFtZTwvdGV4dD4KICA8dGV4dCB4bWw6
c3BhY2U9InByZXNlcnZlIiB4PSI1Ny42MTgiIHk9IjY2LjM0OCIgPjE8L3RleHQ+CiAgPHRleHQg
eG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iMTQxLjQyNjAxIiB5PSI2Ni4zNDgiID5dPC90ZXh0Pgog
IDxsaW5lIHgxPSIxNTMuNjQ4MDEiIHkxPSI2Mi44NTYwMDMiIHgyPSIxNzQuNiIgeTI9IjYyLjg1
NjAwMyIgY2xhc3M9InNvbGlkIj48L2xpbmU+CiAgPHRleHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIg
eD0iMTgzLjMzIiB5PSI2Ni4zNDgiID57Sk1QfTwvdGV4dD4KICA8bGluZSB4MT0iMjIzLjQ4OCIg
eTE9IjYyLjg1NjAwMyIgeDI9IjIzNy40NTYwMSIgeTI9IjYyLjg1NjAwMyIgY2xhc3M9InNvbGlk
Ij48L2xpbmU+CiAgPHBvbHlnb24gcG9pbnRzPSIyMzcuNDU2MDEsNTkuMzY0MDAyIDI0NC40NCw2
Mi44NTYwMDMgMjM3LjQ1NjAxLDY2LjM0OCIgY2xhc3M9ImZpbGxlZCI+PC9wb2x5Z29uPgogIDx0
ZXh0IHhtbDpzcGFjZT0icHJlc2VydmUiIHg9IjI1My4xNzAwMSIgeT0iNjYuMzQ4IiA+TG9vcDwv
dGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIyODguMDkiIHk9IjY2LjM0OCIg
PkhlYWRlcjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzMzYuOTc4IiB5
PSI2Ni4zNDgiID5pbjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIzNTcu
OTMwMDIiIHk9IjY2LjM0OCIgPlBsYWNlPC90ZXh0PgogIDx0ZXh0IHhtbDpzcGFjZT0icHJlc2Vy
dmUiIHg9IjEuNzQ2IiB5PSIxMC40NzYiID5TdGFuZGFyZCBSZWN1cnNpb24gKFN0YWNrIERlcHRo
IE4pOjwvdGV4dD4KICA8dGV4dCB4bWw6c3BhY2U9InByZXNlcnZlIiB4PSIxLjc0NiIgeT0iNTIu
MzgiID5UYWlsLUNhbGwgT3B0aW1pc2F0aW9uIChTdGFjayBEZXB0aCAxKTo8L3RleHQ+CiAgPHRl
eHQgeG1sOnNwYWNlPSJwcmVzZXJ2ZSIgeD0iNzEuNTg2IiB5PSI2Ni4zNDgiID4oUmV1c2VkKTwv
dGV4dD4KPC9zdmc+Cg==
" alt="" />
<figcaption>Figure 7: Standard Recursion versus TCO</figcaption>
</figure>

**Specification:**

Analyse and transform recursive implementations:

1.  Implement a standard, non-tail-recursive `sum_range(uint64_t n)`:
    returns $\sum_{i=1}^n i$.
2.  Implement a tail-recursive version
    `sum_range_tail(uint64_t n, uint64_t accumulator = 0)`.
3.  Test both with $N = 1,000,000$. Verify that the non-tail-recursive
    version crashes with a stack overflow when compiled with `-O0`,
    while the tail-recursive version succeeds under `-O2`/`-O3`.
4.  Inspect the generated assembly to verify that the compiler replaced
    the recursive call with an unconditional backward jump (`jmp`).

``` cpp
#include <cstdint>

// Non-Tail Recursive: Work remains after call returns
uint64_t sum_non_tail(uint64_t n) {
    if (n == 0) return 0;
    return n + sum_non_tail(n - 1);
}

// Tail-Recursive: Work passed forward into accumulator
uint64_t sum_tail(uint64_t n, uint64_t acc = 0) {
    if (n == 0) return acc;
    return sum_tail(n - 1, acc + n); // Compiler converts to JMP!
}
```

**Verification Tasks:**

- **Task 1 (Mathematical Accuracy):** Verify that for $N = 100$, both
  functions return $\frac{100 \times 101}{2} = 5050$.
- **Task 2 (Stack Overflow Demonstration):** Run
  `sum_non_tail(1'000'000)` under debug mode (`-O0`). Confirm that a
  stack overflow segmentation fault occurs.
- **Task 3 (TCO Assembly Verification):** Compile `sum_tail` with `-O2`
  on Compiler Explorer. Verify that the output contains zero `call`
  instructions, executing entirely using register additions and a `jne`
  loop.